# Diagnóstico local de observabilidad desde CO₂ — LAB019–LAB021

**SOURCE:** este archivo conserva código y explicaciones sin salidas. Su pareja
`co2_local_observability.executed.ipynb` conserva la ejecución completa en Conda `fermentation`.

**Pregunta:** ¿qué cambios en los estados dejan efectos distinguibles en la historia del caudal de CO₂,
y cuánto depende esa información de conocer la capa de transferencia y la ganancia?
No se implementa un EKF, no se ajustan parámetros ni se certifica observabilidad global.

Se trabaja con la **referencia simplificada** de la auditoría térmica: theta_full, capa CO₂ histórica,
transitorio dentro de q_bio y sin término respiratorio explícito. Se conserva el O₂ auxiliar.
Las funciones y lectores necesarios están incluidos aquí: no se ejecuta otro notebook ni sus optimizadores.

La ejecución separa: (1) reconstrucción y error de la referencia; (2) sensibilidad de condiciones
iniciales desde t₀, con gate nominal fijado; (3) estados al comienzo de ventanas pre-nutrición,
una vez terminado el gate; (4) pérdida de información al admitir parámetros inciertos;
(5) aporte hipotético de muestras offline. Los datos procesados y T contienen filtrado retrospectivo:
este es un diagnóstico offline, no una demostración de funcionamiento online.

**Ampliación:** la sección 11 compara `theta_full` y `theta_defaults` con la capa de **parámetros antiguos del notebook 02**. Incorpora priors exploratorios, información secuencial del CO₂, una muestra Y15 y comprobaciones no lineales. Las secciones anteriores conservan el diagnóstico de referencia con `theta_full`; sus rangos no sustituyen el análisis con priors.

In [ ]:
from __future__ import annotations

import csv
import io
import json
import math
import re
import sys
from dataclasses import replace
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from IPython import get_ipython
_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("matplotlib", "inline")
import numpy as np
import openpyxl
import pandas as pd
from IPython.display import Markdown, Image, display

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "fermentation_model").exists())
FM = ROOT / "fermentation_model"
LAB2026 = FM / "laboratory_2026"
DATA_DIR = FM / "data" / "mem2026" / "LAB019-021"
DATA_016_018_DIR = FM / "data" / "mem2026" / "LAB016-018"
REFERENCE_RESULTS_DIR = LAB2026 / "results" / "lab019_021_theta_full_holdout"

for path in (FM, LAB2026, FM / "pilot_2025"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from shared import run_new_must_glycerol_estimability_doe as base
from shared import run_secondary_joint_campaign_doe as joint
from laboratory_2026 import run_estimability_historical_by_medium as natural_loader
from laboratory_2026 import run_co2_matrix_cross_validation_2026 as co2_cross

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9.5, "font.family": "DejaVu Sans"})


def display_inline_figure(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', dpi=110, bbox_inches='tight', pad_inches=.15, facecolor='white')
    plt.close(fig)
    display(Image(data=buffer.getvalue()))
    buffer.close()

LABS = ("LAB019", "LAB020", "LAB021")
FERMENTER = {"LAB019": "F1", "LAB020": "F2", "LAB021": "F3"}
MATCHED_ORIGINAL = {"LAB019": "LAB004", "LAB020": "LAB006", "LAB021": "LAB009"}
COLORS = {"LAB019": "#4c72b0", "LAB020": "#55a868", "LAB021": "#c44e52"}
T0 = pd.Timestamp("2026-09-21 17:00:00")
# Hora y dosis reales del segundo pulso: hoja Nutricion, no un valor heredado del ajuste histórico.
nutrition_wb = openpyxl.load_workbook(
    DATA_DIR / "Mediciones-Offline-LAB019-021.xlsx", read_only=True, data_only=True
)
pulse2_rows = [
    row for row in nutrition_wb["Nutricion"].iter_rows(min_row=2, values_only=True)
    if row[0] in LABS and row[2] == "Pulso 2"
]
nutrition_wb.close()
assert len(pulse2_rows) == len(LABS)
assert {row[0] for row in pulse2_rows} == set(LABS)
PULSE2_DT = {row[0]: pd.Timestamp(row[3]) for row in pulse2_rows}
PULSE2_H = {lab: (stamp - T0).total_seconds() / 3600.0 for lab, stamp in PULSE2_DT.items()}
pulse2_doses = {row[0]: float(row[4]) / 1000.0 for row in pulse2_rows}  # mg/L -> kg/m³
assert len(set(pulse2_doses.values())) == 1, "el modelo usa una dosis común para los tres lotes"
DELTA_N_KG_M3 = next(iter(pulse2_doses.values()))
GRID_H = 0.25
CO2_FACTOR_G_L_H_PER_SCCM = 44.0095 * 60.0 * 0.74 / (1000.0 * 24.16 * 2.0)
assert np.isclose(CO2_FACTOR_G_L_H_PER_SCCM, 0.0404391928807947)
print("ROOT:", ROOT)
print("t0:", T0, "| pulse 2 [h]:", PULSE2_H)

pd.set_option("display.max_rows", 120)

assert Path(sys.prefix).name == 'fermentation', sys.executable
print('Python:', sys.executable)

import hashlib
import time
from scipy.integrate import solve_ivp
from scipy.linalg import svd
OUT = LAB2026 / 'notebooks' / 'EKF' / 'observability_results'
OUT.mkdir(exist_ok=True)
RECENT_LABS = LABS
CONFIG = dict(fd_fraction=.01, fd_check_fraction=.005, primary_cadence_h=3.,
              sigma_floor=.04, sigma_relative_peak=.10, rank_rtol=1e-8, rank_atol=1e-6,
              primary_grid_h=.25, rtol=2e-9, atol=2e-11, max_step_h=.5)
STATE_NAMES = ('X','N','G','F','E','Cd','O2','Xd','Gly')
TARGET_NAMES = STATE_NAMES[:7]
# Escalas de incertidumbre exploratorias, no desviaciones estimadas de nuestros instrumentos.
STATE_SCALE = dict(X=.05,N=.025,G=5.,F=5.,E=2.,Cd=.10,O2=.20,Xd=.05,Gly=.5)
NUISANCE_NAMES = ('matrix_gain','kCO2_release_h','CO2sat_scale','bio_scale')
UNITS = dict(X='g/L',N='g N/L',G='g/L',F='g/L',E='g/L',Cd='g CO2/L',O2='mg O2/L',Xd='g/L',Gly='g/L')
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024*1024), b''): digest.update(block)
    return digest.hexdigest()
PROTECTED = [p for p in DATA_DIR.rglob('*') if p.is_file()] + [
    LAB2026/'results/estimability_historical_natural/theta_natural_full.csv',
    LAB2026/'results/estimability_historical_natural/theta.csv',
    LAB2026/'results/co2_matrix_cross_validation_2026/fit_parameters.csv',
    LAB2026/'notebooks/diagnostics/lab019_021_co2_temperature_shape_audit.ipynb',
    LAB2026/'notebooks/co2_reformulation/02_nutrition_structure_comparison.source.ipynb',
    FM/'shared/run_new_must_glycerol_estimability_doe.py',
    FM/'shared/run_secondary_joint_campaign_doe.py',
    LAB2026/'run_co2_matrix_cross_validation_2026.py',
    FM/'pilot_2025/run_pilot_2025_co2_solubility_integrated_doe.py']
HASH_BEFORE = {str(p.relative_to(ROOT)):sha256(p) for p in PROTECTED}
pd.set_option('display.max_rows',40)
display(pd.DataFrame([{'estado':s,'escala_exploratoria':STATE_SCALE[s],'unidad':UNITS[s]} for s in STATE_NAMES]))
display(pd.DataFrame([CONFIG]))

## 1. Parámetros, datos y procedencia

Se reutilizan los lectores de la comparación nutricional y la auditoría térmica, incluidos como
código local. T medida y pulsos registrados son entradas idénticas para todas las perturbaciones.
La química posterior y Oculyze se muestran como comparación; **no se ajusta el modelo a ellas**.
La concentración inicial X conserva la convención de la referencia publicada (Oculyze m1 crudo × 0.03,
sin multiplicar por viabilidad); la comparación posterior usa biomasa viable. Esta diferencia se
documenta y no se corrige silenciosamente. m1 sirve para inicializar y no cuenta como validación.

### Muestras de nutrición NUT

Se incorporan `LAB019-NUT`, `LAB020-NUT` y `LAB021-NUT`, que estaban en las
exportaciones Y15 pero quedaban fuera al seleccionar sólo muestras numeradas.
Contienen PAN, amonio y YAN; los otros canales permanecen ausentes.
Se conserva YAN informado y YAN calculado con PAN + 0,823·max(NH3,0), sin duplicarlos
como mediciones independientes. Los registros YAN repetidos idénticos se deduplican.

La hora de extracción no está en `MedicionesOffline`. Provisionalmente, suponiendo
que NUT se tomó después de la adición, se representa en el punto medio del intervalo
entre el pulso 2 registrado y el primer análisis de N de esa muestra. El intervalo se
dibuja horizontalmente y **no representa una precisión estadística**. La hora de
análisis se conserva separada; no se presenta como una hora de muestreo conocida.
`NUT_SAMPLE_TIMES` permite introducir horas de extracción confirmadas.


NUT se conserva como **contraste reservado**, sin convertirla en otra muestra Y15
completa ni en un ancla adicional. Se muestra en las figuras y en una tabla de
predicción de N que recorre su intervalo temporal. Se excluye de las métricas de
cobertura local que suponen una hora exacta. El pulso de entrada ya estaba incorporado:
añadir NUT no aplica una segunda dosis ni cambia los parámetros antiguos.


In [ ]:
DEFAULT_PARAMETERS = ('sN','qXG','qXF','sG','sF','m0')
def load_parameters(fm):
    results = fm / 'laboratory_2026' / 'results'
    full_path = results / 'estimability_historical_natural/theta_natural_full.csv'
    old_path = results / 'estimability_historical_natural/theta.csv'
    layer_path = results / 'co2_matrix_cross_validation_2026/fit_parameters.csv'
    full_table = pd.read_csv(full_path)
    full = dict(zip(full_table.parameter, full_table.theta.astype(float)))
    old_raw = pd.read_csv(old_path, index_col=0).iloc[:, 0].astype(float).to_dict()
    defaults = co2_cross._load_theta(old_path)
    assert set(full) == set(base.FULL17)
    assert set(base.FULL17) - set(old_raw) == set(DEFAULT_PARAMETERS)
    defaults = {key: float(defaults[key]) for key in base.FULL17}
    for key in base.FULL17:
        expected = base.DEFAULT_THETA[key] if key in DEFAULT_PARAMETERS else full[key]
        assert np.isclose(defaults[key], expected, rtol=1e-12, atol=0), key
    layer = pd.read_csv(layer_path)
    layer = layer[layer.calibration_matrix.eq('natural')].copy()
    params = dict(zip(layer.parameter, layer.estimate.astype(float)))
    assert set(params) == set(co2_cross.PARAMETER_BOUNDS)
    params.update(A=.30, tau_rise_h=4., tau_decay_h=12.,
                  N_ref_g_l=.14, kernel_normalization='grid_max')
    table = pd.DataFrame({'parameter': base.FULL17,
                          'theta_full': [full[k] for k in base.FULL17],
                          'theta_defaults': [defaults[k] for k in base.FULL17],
                          'origen_defaults': ['DEFAULT_THETA' if k in DEFAULT_PARAMETERS else 'calibración natural histórica' for k in base.FULL17]})
    return {'theta_full': full, 'theta_defaults': defaults}, params, table, layer, (full_path, old_path, layer_path)
THETA_SETS, CO2_PARAMS, THETA_TABLE, CO2_PARAMETER_TABLE, _ = load_parameters(FM)
theta = THETA_SETS['theta_full']
CO2_PARAMS['bio_scale'] = 1.
NUISANCE_SCALE = {k:CO2_PARAMS[k]*f for k,f in zip(NUISANCE_NAMES,(.10,.20,.20,.20))}
display(THETA_TABLE)
display(pd.DataFrame([{'parameter':k,'value':v} for k,v in CO2_PARAMS.items()]))
print('Factor CO2/etanol heredado:',joint.CO2_G_PER_G_ETHANOL,
      '| ideal alcohólico:',44.01/46.07,
      '| se conserva la calibración; bio_scale sólo es una hipótesis diagnóstica')

In [ ]:
def read_csv_window(path, usecols, start, end, chunksize=500_000):
    frames = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize, low_memory=False):
        stamp = pd.to_datetime(chunk["timestamp"], errors="coerce")
        keep = stamp.between(start, end)
        if keep.any():
            part = chunk.loc[keep].copy()
            part["timestamp"] = stamp.loc[keep]
            frames.append(part)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=usecols)


window_start, window_end = T0 - pd.Timedelta(hours=6), T0 + pd.Timedelta(hours=170)
temperature_raw = read_csv_window(
    DATA_DIR / "backup_global_temperatura.csv", ["timestamp", "fermentador", "T", "SP"], window_start, window_end
)
for col in ("T", "SP"):
    temperature_raw[col] = pd.to_numeric(temperature_raw[col], errors="coerce")
co2_raw = read_csv_window(
    DATA_DIR / "backup_global_co2.csv", ["timestamp", "fermentador", "flow_sccm", "flow_filt_sccm", "status"], window_start, window_end
)
for col in ("flow_sccm", "flow_filt_sccm"):
    co2_raw[col] = pd.to_numeric(co2_raw[col], errors="coerce")

TEMP_RESAMPLE = "1min"    # mediana por minuto: elimina outliers y uniformiza el cadence
TEMP_MEDIAN = "20min"     # mediana móvil: suprime ráfagas y spikes del control on/off
TEMP_MEAN = "45min"       # media móvil: suaviza la oscilación de banda del control

temp_log = {}
for _lab in LABS:
    _f = FERMENTER[_lab]
    frame = (temperature_raw[temperature_raw["fermentador"].eq(_f)]
             .dropna(subset=["T"]).sort_values("timestamp")
             .set_index("timestamp")[["T", "SP"]])
    frame = frame.resample(TEMP_RESAMPLE).median()
    t_med = frame["T"].rolling(TEMP_MEDIAN, center=True, min_periods=1).median()
    temp_log[_f] = pd.DataFrame({
        "T_filt": t_med.rolling(TEMP_MEAN, center=True, min_periods=1).mean(),
        "SP": frame["SP"],
    })

In [ ]:
from shared.new_must_data_loader import ETHANOL_G_L_PER_PERCENT_VV


def numeric(value):
    try:
        out = float(value)
        return out if np.isfinite(out) else np.nan
    except (TypeError, ValueError):
        return np.nan


def load_offline_samples():
    path = DATA_DIR / "Mediciones-Offline-LAB019-021.xlsx"
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    rows = []
    for excel_row, row in enumerate(wb["MedicionesOffline"].iter_rows(min_row=2, values_only=True), start=2):
        fecha, hora, lab, brix, density = row[0], row[1], row[2], row[5], row[6]
        if lab is None and brix is None and density is None:
            continue
        if lab not in LABS or fecha is None or hora is None or row[4] is None:
            raise ValueError(f"MedicionesOffline fila {excel_row}: metadatos de muestra incompletos")
        hh = hora.time() if isinstance(hora, datetime) else hora
        rows.append({
            "lab": str(lab), "sample": str(row[4]).strip(),
            "sample_timestamp": datetime.combine(fecha.date() if hasattr(fecha, "date") else fecha, hh),
            "Brix": numeric(brix), "density": numeric(density),
            "Etanol_pct_vv": numeric(row[10]), "metadata_fallback": False,
        })
    wb.close()
    result = pd.DataFrame(rows)
    assert not result.duplicated(["lab", "sample"]).any(), "muestras offline duplicadas"
    return result

def load_y15():
    rows = []
    paths = sorted(DATA_DIR.glob("EXPAuto*.txt")) + sorted(DATA_DIR.glob("ONLINE*.txt"))
    for path in paths:
        for line in open(path, encoding="utf-8-sig", errors="replace"):
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 6:
                continue
            raw = fields[3].strip()
            value = float(raw.replace(",", ".")) if re.fullmatch(r"-?\d+(,\d+)?", raw) else np.nan
            rows.append({"sample_id": fields[0], "analyte": fields[1], "value": value, "raw_value": raw, "analysis_timestamp": fields[5], "source_file": path.name})
    long = pd.DataFrame(rows).drop_duplicates(["sample_id", "analyte"], keep="first")
    wide = long.pivot(index="sample_id", columns="analyte", values="value")
    raw_wide = long.pivot(index="sample_id", columns="analyte", values="raw_value")
    return long, wide, raw_wide


offline = load_offline_samples()
# Dilución Alcolyzer m12, distinta por lote (cuaderno de laboratorio): LAB020 27.5 mL de vino + 10 mL de agua;
# LAB021 32.5 mL de vino + 5 mL de agua (ambas a 37.5 mL). Sólo corrige etanol (medido en la mezcla);
# Brix y densidad se midieron antes de diluir.
DILUTION_FACTORS_M12 = {"LAB020": 37.5 / 27.5, "LAB021": 37.5 / 32.5}
for _lab_m12, _factor in DILUTION_FACTORS_M12.items():
    _sel = offline["lab"].eq(_lab_m12) & offline["sample"].eq("12")
    offline.loc[_sel, "Etanol_pct_vv"] = offline.loc[_sel, "Etanol_pct_vv"] * _factor
assert np.isclose(offline.loc[(offline["lab"] == "LAB020") & (offline["sample"] == "12"), "Etanol_pct_vv"].iloc[0], 7.33 * 15 / 11)
assert np.isclose(offline.loc[(offline["lab"] == "LAB021") & (offline["sample"] == "12"), "Etanol_pct_vv"].iloc[0], 8.66 * 15 / 13)
y15_long, y15, y15_raw = load_y15()

chem_rows = []
for lab in LABS:
    samples = offline[offline["lab"].eq(lab)].copy().set_index("sample")
    numbered_samples = sorted((item for item in samples.index if str(item).isdigit()), key=int)
    assert "1" in numbered_samples, f"{lab}: falta muestra inicial"
    for sample in numbered_samples:
        sid = f"{lab}-{sample}"
        meta = samples.loc[sample]
        stamp = pd.Timestamp(meta["sample_timestamp"])
        sample_t_h = (stamp - T0).total_seconds() / 3600.0
        t_h = 0.0 if sample == "1" else sample_t_h
        get = lambda name: float(y15.loc[sid, name]) if sid in y15.index and name in y15.columns and pd.notna(y15.loc[sid, name]) else np.nan
        get_raw = lambda name: str(y15_raw.loc[sid, name]) if sid in y15_raw.index and name in y15_raw.columns and pd.notna(y15_raw.loc[sid, name]) else ""
        gf_320_raw = get("GLU-FRU-320")
        gf_low_raw = get("GLUCOSE-FRUCTOSE")
        if np.isfinite(gf_320_raw) and gf_320_raw >= 0.0:
            gf, gf_source = gf_320_raw, "GLU-FRU-320"
        elif np.isfinite(gf_low_raw) and gf_low_raw >= 0.0:
            gf, gf_source = gf_low_raw, "GLUCOSE-FRUCTOSE"
        else:
            gf, gf_source = np.nan, ""
        glucose_320_raw = get("GLUCOSE-320")
        glucose_low_raw = get("GLUCOSE")
        glucose_raw = glucose_320_raw if np.isfinite(glucose_320_raw) else glucose_low_raw
        glucose = glucose_raw if np.isfinite(glucose_raw) and glucose_raw >= 0.0 else np.nan
        fructose = gf - glucose if np.isfinite(gf) and np.isfinite(glucose) and gf >= glucose else np.nan
        pan, ammonia_raw = get("PAN"), get("AMMONIA")
        ammonia = max(ammonia_raw, 0.0) if np.isfinite(ammonia_raw) else np.nan
        yan = pan + 0.823 * ammonia if np.isfinite(pan) and np.isfinite(ammonia) else np.nan
        chem_rows.append({
            "lab": lab, "sample": sample, "sample_timestamp": stamp, "sample_t_h": sample_t_h, "t_h": t_h,
            "metadata_fallback": bool(meta["metadata_fallback"]), "density": float(meta["density"]), "Brix": float(meta["Brix"]),
            "GF_g_l": gf, "GF_source": gf_source,
            "GF_320_raw_g_l": gf_320_raw, "GF_low_raw_g_l": gf_low_raw,
            "GF_320_below_zero": bool(np.isfinite(gf_320_raw) and gf_320_raw < 0.0),
            "G_g_l": glucose, "G_raw_g_l": glucose_raw, "F_g_l": fructose,
            "PAN_mg_l": pan, "NH3_raw_mg_l": ammonia_raw, "NH3_censored_mg_l": ammonia,
            "YAN_mg_l": yan, "N_kg_m3": yan / 1000.0 if np.isfinite(yan) else np.nan,
            "Gly_g_l": get("GLYCEROL"), "Pyr_mg_l": get("PYRUVIC ACID"),
            "Etanol_pct_vv": float(meta["Etanol_pct_vv"]),
            "E_g_l": meta["Etanol_pct_vv"] * ETHANOL_G_L_PER_PERCENT_VV if np.isfinite(meta["Etanol_pct_vv"]) else np.nan,
            "AcAld_mg_l": get("ACETALDEHID-1000"), "Acetate_g_l": get("ACETIC ACID"),
            "G_raw_text": get_raw("GLUCOSE-320") or get_raw("GLUCOSE"),
            "GF_raw_text": get_raw("GLU-FRU-320"),
            "GF_low_raw_text": get_raw("GLUCOSE-FRUCTOSE"),
            "NH3_below_zero": bool(np.isfinite(ammonia_raw) and ammonia_raw < 0.0),
        })
from laboratory_2026.nut_sample_audit import load_nut_samples
NUT_SAMPLE_TIMES = {}  # completar sólo con horas de extracción confirmadas
NUT_SAMPLES = load_nut_samples(DATA_DIR, {lab:T0 for lab in LABS}, NUT_SAMPLE_TIMES)
NUT_SAMPLES.to_csv(OUT/'nut_sample_audit.csv', index=False)
for nut in NUT_SAMPLES.itertuples():
    chem_rows.append(dict(lab=nut.lab, sample='NUT', sample_timestamp=nut.sample_timestamp,
        sample_t_h=nut.plot_t_h, t_h=nut.plot_t_h, metadata_fallback=nut.time_provisional,
        PAN_mg_l=nut.PAN_mg_l, NH3_raw_mg_l=nut.NH3_raw_mg_l,
        NH3_censored_mg_l=nut.NH3_censored_mg_l, YAN_mg_l=nut.YAN_mg_l, N_kg_m3=nut.N_kg_m3,
        YAN_reported_mg_l=nut.YAN_reported_mg_l, NH3_below_zero=False,
        time_lower_h=nut.time_lower_h, time_upper_h=nut.time_upper_h,
        time_provisional=nut.time_provisional, time_source=nut.time_source))
chem = pd.DataFrame(chem_rows).sort_values(["lab", "t_h"]).reset_index(drop=True)
chem['is_NUT'] = chem['sample'].eq('NUT')
chem['time_provisional'] = chem['time_provisional'].fillna(False).astype(bool)
chem['time_lower_h'] = chem['time_lower_h'].fillna(chem.t_h)
chem['time_upper_h'] = chem['time_upper_h'].fillna(chem.t_h)
chem['time_source'] = chem['time_source'].fillna('offline extraction timestamp; m1 mapped to t0')
chem["is_initial"] = chem["sample"].eq("1")
chem["soft_flag_GF"] = chem["lab"].eq("LAB019") & chem["sample"].eq("4")
chem["soft_flag_metadata"] = chem["metadata_fallback"]
chem["y15_available"] = chem[["GF_g_l", "YAN_mg_l", "Gly_g_l", "Pyr_mg_l", "AcAld_mg_l"]].notna().any(axis=1)
assert chem[chem["sample"].eq("1")]["E_g_l"].notna().all(), "falta Etanol m1 (inicializa E0)"

assert np.allclose(chem[chem["sample"].eq("1")]["t_h"], 0.0)
assert chem.query("lab == 'LAB020' and sample == '4'")["G_g_l"].isna().all()
assert chem.query("lab == 'LAB020' and sample == '4'")["G_raw_g_l"].iloc[0] == -7.28
assert chem[chem["sample"].eq("12")]["y15_available"].all(), "falta exportación Y15 m12"
assert chem.loc[chem["sample"].eq("12"), "GF_320_below_zero"].all(), "revisar QA de GF m12"
display(chem[["lab", "sample", "sample_t_h", "t_h", "GF_g_l", "G_g_l", "F_g_l", "YAN_mg_l", "Gly_g_l", "E_g_l", "Pyr_mg_l", "AcAld_mg_l", "y15_available", "NH3_below_zero", "metadata_fallback"]])
print("Muestras sin resultado Y15:", chem.loc[~chem["y15_available"], ["lab", "sample"]].to_dict("records"))

# Todas las mediciones Oculyze: concentración total en millones de células/mL,
# conversión de 30 pg/célula y viabilidad porcentual para comparar con X viable.
from shared.new_must_data_loader import MILLION_CELLS_ML_TO_KG_M3
biomass_rows=[]
for lab in LABS:
    report_path=DATA_DIR/f'Oculyze_report_{lab}.csv'
    with open(report_path,encoding='utf-8-sig') as handle:
        report_rows=list(csv.DictReader(handle))
    for row in report_rows:
        sample_id=row['Name'].strip().upper()
        if sample_id=='CERO':
            continue  # Se conserva la convención primaria de concentración cruda.
        match=re.fullmatch(rf'{lab}-(\d+)',sample_id)
        if match is None:
            raise ValueError(f'{report_path.name}: identificador inesperado {sample_id}')
        sample=str(int(match.group(1)))
        meta=chem[chem.lab.eq(lab)&chem['sample'].eq(sample)]
        assert len(meta)==1,(lab,sample)
        meta=meta.iloc[0]
        concentration=float(row['Concentration'])
        viability=float(row['Viability'])
        assert np.isfinite(concentration) and concentration>=0
        assert np.isfinite(viability) and 0<=viability<=100
        total=concentration*MILLION_CELLS_ML_TO_KG_M3
        viable=total*viability/100.
        biomass_rows.append(dict(lab=lab,sample=sample,sample_id=sample_id,
            sample_timestamp=meta.sample_timestamp,sample_t_h=float(meta.sample_t_h),
            t_h=float(meta.t_h),concentration_mcells_ml=concentration,viability_pct=viability,
            X_total_g_l=total,X_viable_g_l=viable,Xd_g_l=total-viable,
            source_file=report_path.name))
BIOMASS_OBSERVATIONS=pd.DataFrame(biomass_rows).sort_values(['lab','t_h']).reset_index(drop=True)
assert not BIOMASS_OBSERVATIONS.duplicated(['lab','sample']).any()
assert BIOMASS_OBSERVATIONS.groupby('lab').size().to_dict()=={lab:11 for lab in LABS}
assert np.allclose(BIOMASS_OBSERVATIONS.X_total_g_l,
                   BIOMASS_OBSERVATIONS.X_viable_g_l+BIOMASS_OBSERVATIONS.Xd_g_l)
chem=chem.merge(BIOMASS_OBSERVATIONS[['lab','sample','X_total_g_l','X_viable_g_l','Xd_g_l']],
                on=['lab','sample'],how='left',validate='one_to_one')
display(BIOMASS_OBSERVATIONS[['lab','sample','sample_t_h','t_h','concentration_mcells_ml',
                             'viability_pct','X_total_g_l','X_viable_g_l','Xd_g_l']].round(5))
print('Mediciones Oculyze por lote:',BIOMASS_OBSERVATIONS.groupby('lab').size().to_dict())

assert chem[chem.is_NUT].lab.nunique() == 3
assert chem.loc[chem.is_NUT, ['GF_g_l','G_g_l','F_g_l','Gly_g_l','E_g_l','X_viable_g_l']].isna().all().all()
chem.to_csv(OUT/'chemistry_samples_with_NUT.csv', index=False)
display(NUT_SAMPLES[['lab','YAN_reported_mg_l','YAN_mg_l','pulse_timestamp',
                     'analysis_PAN','analysis_YAN','sample_timestamp','time_lower_h','plot_t_h','time_upper_h']])
def draw_nut_state(ax, lab, state):
    if state != 'N': return
    nut = NUT_SAMPLES[NUT_SAMPLES.lab.eq(lab)].iloc[0]
    x = float(nut.plot_t_h)
    ax.errorbar([x],[nut.N_kg_m3],
        xerr=[[x-nut.time_lower_h],[nut.time_upper_h-x]],fmt='*',color='#8e24aa',
        ms=8,capsize=3,zorder=8,label='NUT · tiempo provisional' if nut.time_provisional else 'NUT · hora confirmada')


In [ ]:
co2_hourly_parts, mask_rows = [], []
for lab in LABS:
    f = FERMENTER[lab]
    frame = co2_raw[co2_raw["fermentador"].eq(f)].sort_values("timestamp").copy()
    frame["t_h"] = (frame["timestamp"] - T0).dt.total_seconds() / 3600.0
    baseline = frame[frame["t_h"].between(-2.0, -0.5)]["flow_sccm"].dropna().to_numpy(dtype=float)
    baseline_center = float(np.median(baseline))
    sigma = float(1.4826 * np.median(np.abs(baseline - baseline_center)))
    threshold = max(3.5 * sigma, 0.5)
    search = frame[frame["t_h"].between(-0.5, 0.75)].copy()
    crossing = search[np.abs(search["flow_sccm"] - baseline_center) > threshold]
    if crossing.empty:
        event_start_h, event_raw_end_h = 0.0, 1.0
    else:
        event_start_h, event_raw_end_h = float(crossing["t_h"].min()), float(crossing["t_h"].max())
    mask_end_h = max(0.0, event_raw_end_h) + 1.5

    zero_support = frame[frame["t_h"].between(mask_end_h, 12.0)]["flow_filt_sccm"].dropna()
    zero_sccm = max(0.0, float(zero_support.quantile(0.10))) if len(zero_support) else 0.0
    frame["flow_corrected_sccm"] = np.maximum(frame["flow_filt_sccm"] - zero_sccm, 0.0)
    hourly = frame.set_index("timestamp")[["flow_corrected_sccm", "flow_filt_sccm", "flow_sccm"]].resample("1h").median().dropna(subset=["flow_corrected_sccm"]).reset_index()
    hourly["t_h"] = (hourly["timestamp"] - T0).dt.total_seconds() / 3600.0
    hourly = hourly[hourly["t_h"].between(0.0, 170.0)].copy()
    hourly["lab"] = lab
    hourly["co2_observed_g_l_h"] = hourly["flow_corrected_sccm"] * CO2_FACTOR_G_L_H_PER_SCCM
    hourly["co2_valid"] = hourly["t_h"] > mask_end_h
    hourly["left_censored"] = hourly["co2_observed_g_l_h"] < co2_cross.CO2_DETECTION_LIMIT_G_L_H
    co2_hourly_parts.append(hourly)
    mask_rows.append({
        "lab": lab, "fermenter": f, "baseline_raw_sccm": baseline_center, "baseline_sigma_sccm": sigma,
        "artifact_threshold_sccm": threshold, "event_start_h": event_start_h,
        "event_raw_end_h": event_raw_end_h, "recovery_tail_min": 90, "mask_end_h": mask_end_h,
        "sensor_zero_sccm": zero_sccm,
    })

co2_hourly = pd.concat(co2_hourly_parts, ignore_index=True)
mask_audit = pd.DataFrame(mask_rows)
display(mask_audit.round(4))
from shared.new_must_data_loader import MILLION_CELLS_ML_TO_KG_M3
def oculyze_m1_biomass(lab):
    with open(DATA_DIR / f"Oculyze_report_{lab}.csv", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    m1 = next(r for r in rows if r["Name"].strip().upper() == f"{lab}-1")
    cero = next((r for r in rows if r["Name"].strip().upper() == "CERO"), None)
    concentration = float(m1["Concentration"])
    viability = float(m1["Viability"])
    return {
        "x0_kg_m3": concentration * MILLION_CELLS_ML_TO_KG_M3,
        "xd0_kg_m3": concentration * MILLION_CELLS_ML_TO_KG_M3 * (1.0 - viability / 100.0),
        "oculyze_m1_concentration_mcells_ml": concentration,
        "oculyze_m1_viability_pct": viability,
        "x0_cero_subtracted_kg_m3": (
            max(concentration - float(cero["Concentration"]), 0.0) * MILLION_CELLS_ML_TO_KG_M3
            if cero is not None else float("nan")
        ),
    }

recent_batches = {}
recent_time_grids = {}
recent_initial_rows = []
for lab in RECENT_LABS:
    measured = chem[chem.lab.eq(lab)].sort_values('t_h')
    init = measured[measured['sample'].eq('1')].iloc[0]
    biomass = oculyze_m1_biomass(lab)
    support = co2_hourly[co2_hourly.lab.eq(lab)].sort_values('t_h')
    horizon = float(support.t_h.max())
    batch_horizon = max(horizon, float(measured.t_h.max()))
    grid = np.arange(0., batch_horizon + GRID_H / 2., GRID_H)
    grid = np.unique(np.r_[grid, measured.loc[~measured.is_NUT, "t_h"].to_numpy(float), PULSE2_H[lab]])
    grid = grid[(grid >= 0.) & (grid <= batch_horizon + 1e-9)]
    temp = temperature_raw[temperature_raw.fermentador.eq(FERMENTER[lab])].dropna(subset=['T']).sort_values('timestamp')
    temp_h = (temp.timestamp - T0).dt.total_seconds().to_numpy() / 3600.
    temperature = np.interp(grid, temp_h, temp['T'].to_numpy(float))
    initials = {'X': biomass['x0_kg_m3'], 'Xd': biomass['xd0_kg_m3'],
        'N': float(init.N_kg_m3), 'G': float(init.G_g_l), 'F': float(init.F_g_l),
        'E': float(init.E_g_l), 'Gly': float(init.Gly_g_l)}
    assert np.isfinite(list(initials.values())).all()
    recent_batches[lab] = base.BatchData(medium='natural',batch=lab,time=grid,
        temperature_c=temperature, initials=initials,
        pulses={**{channel: tuple() for channel in base.INPUT_CHANNELS}, 'N': ((PULSE2_H[lab],DELTA_N_KG_M3),)},
        observations={state: np.full(len(grid),np.nan) for state in base.STATE_NAMES})
    cache_time = np.arange(0.,horizon+co2_cross.DRIVER_GRID_H+1e-9,co2_cross.DRIVER_GRID_H)
    cache_time = cache_time[cache_time <= horizon+1e-9]
    if not np.isclose(cache_time[-1],horizon): cache_time = np.append(cache_time,horizon)
    recent_time_grids[lab] = cache_time
    recent_initial_rows.append({'lab':lab, **initials, 'X_source':'Oculyze m1 crudo × 0.03',
        'chemistry_source':'Y15/Alcolyzer m1', 'temperature_extrapolated_h':max(0.,batch_horizon-float(temp_h.max()))})
display(pd.DataFrame(recent_initial_rows))

## 2. Referencia y balances

Se mantiene la estructura publicada, incluidos los dos recorridos upstream con/sin pulso N:

\[
q_{bio}=M_N(t)\max\{[1+(g_N-1)r]q_{bio,0}+r(q_{bio,N}-q_{bio,0}),0\},\qquad
q_{prod}=A_{gate}(t)f_{ferm}(O_2)q_{bio}.
\]
\[
q_{gas}=\min\left\{k C_d\left(0.05+0.95\frac{C_d}{C_d+C^*}\right),\,
C_d/\Delta t+q_{prod}\right\},\quad
C_{d,i}=C_{d,i-1}+\Delta t(q_{prod,i-1}-q_{gas,i}),\quad y=\gamma q_{gas}.
\]

La limitación por disponibilidad, los índices de la actualización y el cero del primer punto
son parte del algoritmo histórico; no se sustituyen por un kLa físico.
Se integra upstream con tolerancias más estrictas para diferenciarlo numéricamente y se compara
con la implementación original. El balance del pool y los saltos N se verifican por separado.
Xd y Gly admiten una traslación constante exacta, pues no retroalimentan las tasas. Se integran
sus incrementos desde cero y después se suman sus condiciones iniciales: esto impide que cambiar
una variable sin influencia altere artificialmente los pasos adaptativos del integrador.

In [ ]:
CHEMISTRY_SUGAR_DROP_G_L = 6.
CHEMISTRY_ETHANOL_RISE_G_L = 2.
def integrate_core(batch, theta, t):
    t = np.asarray(t,float)
    state = base.initial_vector(batch).copy()
    offset_indices=[base.STATE_NAMES.index(s) for s in ('Xd','Gly')]
    constant_offsets=state[offset_indices].copy()
    state[offset_indices]=0.
    events = base._pulse_events(batch,float(t[0]),float(t[-1]))
    records = {float(t[0]):state.copy()}
    current = float(t[0])
    if current in events: state += events[current]
    jump_error = 0.
    for boundary in sorted(set([float(t[-1]), *[e for e in events if e>current]])):
        requested = t[(t>current)&(t<=boundary)]
        evaluation = np.unique(np.r_[requested,boundary])
        sol = solve_ivp(lambda ti,yi:base.rhs(ti,yi,theta,batch), (current,boundary),state,
                        t_eval=evaluation,method='LSODA',rtol=CONFIG['rtol'],atol=CONFIG['atol'],
                        max_step=CONFIG['max_step_h'])
        if not sol.success: raise RuntimeError(sol.message)
        for ti,yi in zip(sol.t,sol.y.T): records[float(ti)]=yi.copy()
        state=sol.y[:,-1].copy(); current=boundary
        if boundary in events:
            before=state.copy(); state+=events[boundary]
            jump_error=max(jump_error,float(np.max(np.abs(state-before-events[boundary]))))
    values=np.vstack([records[float(ti)] for ti in t])
    values[:,offset_indices]+=constant_offsets
    assert np.isfinite(values).all(), 'nonfinite core trajectory'
    assert values.min()>-1e-7, f'negative core state: {base.STATE_NAMES[int(np.unravel_index(np.argmin(values),values.shape)[1])]} min={values.min():.6g}'
    assert jump_error<1e-12
    return pd.DataFrame(np.maximum(values,0.),index=t,columns=base.STATE_NAMES)
def simulate_clock(batch, theta, t, B, config):
    assert B==0., 'Este diagnóstico no incorpora relojes cinéticos ajustados.'
    return integrate_core(batch,theta,t)
def speed_factor(temperature, B, reference=20.):
    return np.exp(float(B) * (1. / (reference + 273.15) - 1. / (np.asarray(temperature) + 273.15)))

def cumulative_clock(t, speed):
    return np.r_[0., np.cumsum(.5 * (speed[:-1] + speed[1:]) * np.diff(t))]

def bracket_from_core(core):
    t = core.index.to_numpy(float)
    sugar = (core.G + core.F).to_numpy(float)
    ethanol = core.E.to_numpy(float)
    crossed = np.flatnonzero((sugar <= sugar[0] - CHEMISTRY_SUGAR_DROP_G_L) | (ethanol >= ethanol[0] + CHEMISTRY_ETHANOL_RISE_G_L))
    if not len(crossed):
        return float(t[0]), float(t[0])
    i = int(crossed[0])
    return float(t[max(i - 1, 0)]), float(t[i])

def pulse_multiplier(t, schedule, params):
    result = np.ones(len(t))
    for event, dose in schedule:
        u = np.maximum(t - event, 0.)
        raw = (1. - np.exp(-u / params['tau_rise_h'])) * np.exp(-u / params['tau_decay_h'])
        peak = float(raw.max())
        if peak <= 0:
            raise ValueError('El horizonte no contiene el transitorio nutricional')
        result += params['A'] * dose / params['N_ref_g_l'] * raw / peak
    return result

def oxygen_fraction(t, biomass, initial_saturation, params):
    oxygen = np.zeros(len(t))
    oxygen[0] = params['O2_initial_scale'] * initial_saturation
    for i, dt in enumerate(np.diff(t)):
        uptake = params['O2_qmax_mg_gdw_h'] * biomass[i] * oxygen[i] / (co2_cross.O2_K_MG_L + oxygen[i])
        oxygen[i + 1] = max(oxygen[i] - dt * uptake, 0.)
    anaerobic = co2_cross.O2_ANA_K_MG_L ** co2_cross.O2_ANA_HILL / (
        co2_cross.O2_ANA_K_MG_L ** co2_cross.O2_ANA_HILL + oxygen ** co2_cross.O2_ANA_HILL)
    return oxygen, co2_cross.O2_CRABTREE_FLOOR + (1. - co2_cross.O2_CRABTREE_FLOOR) * anaerobic

def pool(t, csat, qprod, k):
    c = np.zeros(len(t))
    qout = np.zeros(len(t))
    for i, dt in enumerate(np.diff(t), 1):
        previous = c[i - 1]
        fraction = .05 + .95 * previous / max(previous + csat[i], 1e-12)
        qout[i] = min(k[i] * previous * fraction, previous / dt + qprod[i - 1])
        c[i] = max(previous + dt * (qprod[i - 1] - qout[i]), 0.)
    error = float(np.max(np.abs(np.diff(c) - np.diff(t) * (qprod[:-1] - qout[1:]))))
    assert error < 1e-10 and (c >= 0).all() and (qout >= 0).all()
    return c, qout, error

def production_on_core_grid(th,batch,core,t):
    # Same official rate equations, without repeated DataFrame interpolation on an identical grid.
    assert np.array_equal(core.index.to_numpy(float),np.asarray(t,float))
    temperature=np.interp(t,batch.time,batch.temperature_c)
    rows=core[['X','N','G','F','E']].to_numpy(float)
    result=[]
    for temp,(x,n,g,f,e) in zip(temperature,rows):
        terms=base.kinetic_terms(th,float(temp),x,n,g,f,e)
        result.append(joint.CO2_G_PER_G_ETHANOL*max((terms['beta_g']+terms['beta_f'])*max(x,0.),0.))
    return np.asarray(result)

def make_profile(batch, theta, t, params, B, config):
    without = replace(batch, pulses={**batch.pulses, 'N': ()})
    core = simulate_clock(batch, theta, t, B, config)
    core0 = simulate_clock(without, theta, t, B, config)
    temperature = np.array([base.temperature_at(batch, ti) for ti in t])
    speed = speed_factor(temperature, B, config['reference_temperature_C'])
    qn = speed * production_on_core_grid(theta, batch, core, t)
    q0 = speed * production_on_core_grid(theta, without, core0, t)
    schedule = batch.pulses.get('N', ())
    ramp = np.clip((t - schedule[0][0]) / params['pulse_t_rise_h'], 0., 1.) if schedule else np.zeros(len(t))
    biomass = np.maximum(core0.X.to_numpy() + ramp * (core.X.to_numpy() - core0.X.to_numpy()), 0.)
    e, g, f = [np.maximum(core[state].to_numpy(float), 0.) for state in ('E', 'G', 'F')]
    csat = params['CO2sat_scale'] * 1.69 * np.exp(-.032 * (temperature - 20.)) * np.exp(.0016 * e) * np.exp(-.0012 * (g + f))
    o2sat0 = co2_cross.co2_model.o2_saturation_mg_l(temperature[0], e[0], g[0], f[0], 1.)
    oxygen, fferm = oxygen_fraction(t, biomass, o2sat0, params)
    lower, upper = bracket_from_core(core)
    activation = co2_cross._bounded_smoothstep_activation(t, lower, upper,
        params['chem_activation_start_fraction'], params['chem_activation_duration_fraction'])
    return dict(t=t, temperature=temperature, speed=speed,
                clock=cumulative_clock(t, speed), core=core, core0=core0,
                qn=np.asarray(qn), q0=np.asarray(q0), ramp=ramp,
                activity=1. + (params['pulse_activity_gain'] - 1.) * ramp,
                multiplier=pulse_multiplier(t, schedule, params), csat=csat,
                oxygen=oxygen, fferm=fferm, activation=activation,
                lower=lower, upper=upper)

def curve(profile, params, thermal, mode='post', reference=20., temperature_override=None):
    t = profile['t']
    temperature = profile['temperature'] if temperature_override is None else np.full(len(t), float(temperature_override))
    if mode == 'post':
        speed = speed_factor(temperature, thermal.get('B_clock_extra_K', 0.), reference)
        clock = cumulative_clock(t, speed)
        # Same posterior-clock experiment as the original audit. Endpoint holding
        # is explicit and its support exceedance is reported for every candidate.
        q0 = speed * np.interp(clock, t, profile['q0'])
        delta_n = profile['qn'] - profile['q0']
        support_exceeded = clock > t[-1] + 1e-9
    else:
        speed, clock, q0 = profile['speed'], profile['clock'], profile['q0']
        delta_n = profile['qn'] - profile['q0']
        support_exceeded = np.zeros(len(t), dtype=bool)
    adaptation = (1. - np.exp(-clock / float(thermal['tau_adapt_ref_h']))) if 'tau_adapt_ref_h' in thermal else np.ones(len(t))
    qbio = profile['multiplier'] * np.maximum(profile['activity'] * q0 + profile['ramp'] * delta_n, 0.)
    qprod = profile['activation'] * adaptation * profile['fferm'] * qbio
    k = params['kCO2_release_h'] * speed_factor(temperature, thermal.get('B_release_K', 0.), reference)
    c, qout, error = pool(t, profile['csat'], qprod, k)
    frame = pd.DataFrame(dict(t_h=t, temperature_C=temperature, thermal_speed=speed,
        thermal_clock_h=clock, qbio=qbio, qprod=qprod, qout=qout,
        predicted=params['matrix_gain'] * qout, dissolved_g_l=c,
        csat_g_l=profile['csat'], O2_mg_l=profile['oxygen'], f_ferm=profile['fferm'],
        chemical_activation=profile['activation'], adaptation=adaptation,
        transient_multiplier=profile['multiplier'], k_release_h=k,
        clock_outside_driver=support_exceeded))
    frame.attrs['mass_error'] = error
    return frame
def initial_state(batch, params):
    z={**batch.initials,'Cd':0.}
    temp=base.temperature_at(batch,0.)
    z['O2']=params['O2_initial_scale']*co2_cross.co2_model.o2_saturation_mg_l(
        temp,z['E'],z['G'],z['F'],1.)
    return np.array([z[s] for s in STATE_NAMES])
def pool_general(t,csat,qprod,k,c0):
    c=np.zeros(len(t)); q=np.zeros(len(t)); c[0]=c0
    for i,dt in enumerate(np.diff(t),1):
        prev=c[i-1]
        fraction=.05+.95*prev/max(prev+csat[i],1e-12)
        q[i]=min(k*prev*fraction,prev/dt+qprod[i-1])
        c[i]=max(prev+dt*(qprod[i-1]-q[i]),0.)
    error=float(np.max(np.abs(np.diff(c)-np.diff(t)*(qprod[:-1]-q[1:]))))
    assert error<1e-10 and np.isfinite(c).all()
    return c,q,error
def oxygen_general(t,biomass,o0,params):
    o=np.zeros(len(t)); o[0]=o0
    for i,dt in enumerate(np.diff(t)):
        uptake=params['O2_qmax_mg_gdw_h']*biomass[i]*o[i]/(co2_cross.O2_K_MG_L+o[i])
        o[i+1]=max(o[i]-dt*uptake,0.)
    phi=co2_cross.O2_ANA_K_MG_L**co2_cross.O2_ANA_HILL/(co2_cross.O2_ANA_K_MG_L**co2_cross.O2_ANA_HILL+o**co2_cross.O2_ANA_HILL)
    return o,co2_cross.O2_CRABTREE_FLOOR+(1.-co2_cross.O2_CRABTREE_FLOOR)*phi
def full_forward(lab,z=None,params=None,theta_override=None,fixed_gate=None,t_override=None):
    params=CO2_PARAMS if params is None else params
    th=theta if theta_override is None else theta_override
    batch=recent_batches[lab]
    t=recent_time_grids[lab] if t_override is None else t_override
    z=initial_state(batch,params) if z is None else np.asarray(z)
    values=dict(zip(STATE_NAMES,z))
    batch=replace(batch,initials={s:values[s] for s in base.STATE_NAMES})
    profile=make_profile(batch,th,t,params,0.,{'reference_temperature_C':20.})
    if fixed_gate is not None: profile['activation']=np.interp(t,fixed_gate[0],fixed_gate[1])
    biomass=np.maximum(profile['core0'].X.to_numpy()+profile['ramp']*(profile['core'].X.to_numpy()-profile['core0'].X.to_numpy()),0.)
    oxygen,fferm=oxygen_general(t,biomass,values['O2'],params)
    qbio=params['bio_scale']*profile['multiplier']*np.maximum(profile['activity']*profile['q0']+profile['ramp']*(profile['qn']-profile['q0']),0.)
    qprod=profile['activation']*fferm*qbio
    dissolved,qgas,error=pool_general(t,profile['csat'],qprod,params['kCO2_release_h'],values['Cd'])
    out=profile['core'].copy()
    out['Cd']=dissolved; out['O2']=oxygen
    out['predicted']=params['matrix_gain']*qgas
    out['qprod']=qprod; out['qgas']=qgas; out['gate']=profile['activation']; out['qbio']=qbio
    out['temperature_C']=profile['temperature']
    out.attrs.update(mass_error=error,gate_bracket=(profile['lower'],profile['upper']))
    return out

NOMINAL={lab:full_forward(lab) for lab in LABS}
GATES={lab:(n.index.to_numpy(),n.gate.to_numpy()) for lab,n in NOMINAL.items()}
REFERENCE_CHECK=[]
# Función original: mismo upstream histórico, tolerancias originales.
for lab in LABS:
    batch=recent_batches[lab]; tt=recent_time_grids[lab]
    original_integrator=simulate_clock
    def simulate_clock(batch,theta,t,B,config):
        result=base.simulate(batch,theta,t)
        assert result is not None
        return result
    pp=make_profile(batch,theta,tt,CO2_PARAMS,0.,{'reference_temperature_C':20.})
    published=curve(pp,CO2_PARAMS,{},mode='kinetic')
    simulate_clock=original_integrator
    delta=np.max(np.abs(published.predicted.to_numpy()-NOMINAL[lab].predicted.to_numpy()))
    assert delta<.005,'La referencia estricta difiere materialmente de la publicada'
    REFERENCE_CHECK.append(dict(lab=lab,max_output_delta_g_l_h=delta,
        mass_balance_max_g_l=NOMINAL[lab].attrs['mass_error'],pulse_h=PULSE2_H[lab],
        gate_bracket=NOMINAL[lab].attrs['gate_bracket']))
display(pd.DataFrame(REFERENCE_CHECK))
# Independently compare the optimized evaluation against the untouched official rate function.
for lab in LABS:
    batch=recent_batches[lab];core=NOMINAL[lab][list(base.STATE_NAMES)];tt=core.index.to_numpy(float)
    reference_rate=co2_cross.co2_model.co2_production_g_l_h(theta,batch,core,tt)
    rate_parity_error=float(np.max(np.abs(reference_rate-production_on_core_grid(theta,batch,core,tt))))
    assert rate_parity_error<1e-12,rate_parity_error
print('Official CO2 production parity verified <1e-12 g/L/h; no equation or parameter changes.')


In [ ]:
fig,axes=plt.subplots(3,2,figsize=(13,10),layout='constrained')
FIT_ROWS=[]
for i,lab in enumerate(LABS):
    n=NOMINAL[lab]; obs=co2_hourly[co2_hourly.lab.eq(lab)&co2_hourly.co2_valid]
    axes[i,0].plot(n.index,n.predicted,label='Referencia congelada')
    axes[i,0].scatter(obs.t_h,obs.co2_observed_g_l_h,s=9,color='black',label='CO2 procesado')
    axes[i,0].axvline(PULSE2_H[lab],ls='--',color='0.6',label='Pulso N')
    axes[i,0].set(title=lab,xlabel='t [h]',ylabel='CO2 [g/L/h]');axes[i,0].legend(fontsize=8)
    axes[i,1].plot(n.index,n.Cd,label='Cd [g/L]')
    axes[i,1].plot(n.index,n.O2,label='O2 [mg/L]')
    axes[i,1].plot(n.index,n.gate,label='Gate [adimensional]')
    axes[i,1].set(xlabel='t [h]',title='Variables auxiliares; distintas unidades');axes[i,1].legend()
    residual=np.interp(obs.t_h,n.index,n.predicted)-obs.co2_observed_g_l_h
    FIT_ROWS.append(dict(lab=lab,n=len(obs),RMSE_g_l_h=float(np.sqrt(np.mean(residual**2))),
        observed_peak=float(obs.co2_observed_g_l_h.max()),predicted_peak=float(n.predicted.max())))
display_inline_figure(fig)
FIT_TABLE=pd.DataFrame(FIT_ROWS);display(FIT_TABLE)

### Comparación independiente de la trayectoria upstream con química y biomasa

Estas mediciones posteriores a m1 no participan en la simulación ni ajustan sus parámetros.
Se excluye del resumen el punto G/F marcado por QC en LAB019-m4 y se omiten valores faltantes.
La biomasa observada es viable; la condición inicial X mantiene la convención histórica antes
descrita. Un desacuerdo aquí limita la interpretación física de las sensibilidades, aunque el
diagnóstico matemático del modelo se pueda ejecutar.

**Calidad experimental:** la biomasa LAB019–021 es poco confiable según el responsable experimental. Sus puntos se muestran sólo como contexto y se excluyen de las métricas de validación; Y15 y CO₂ son los soportes principales.

In [ ]:
STATE_FIT=[]
OBS_COLUMNS=dict(X='X_viable_g_l',N='N_kg_m3',G='G_g_l',F='F_g_l',E='E_g_l')
fig,axes=plt.subplots(3,5,figsize=(16,10),layout='constrained')
for i,lab in enumerate(LABS):
    nominal=NOMINAL[lab]
    for j,(state,col) in enumerate(OBS_COLUMNS.items()):
        support=chem[chem.lab.eq(lab)&~chem.is_initial&chem[col].notna()].copy()
        if state in ('G','F'):support=support[~support.soft_flag_GF]
        support=support[support.t_h.between(nominal.index.min(),nominal.index.max())]
        plotted_support = support.copy()
        support = support[~support.is_NUT]  # NUT se evalúa separadamente sobre su intervalo.
        predicted=np.interp(support.t_h,nominal.index,nominal[state])
        error=predicted-support[col].to_numpy()
        STATE_FIT.append(dict(lab=lab,state=state,n=len(support),
            RMSE=float(np.sqrt(np.mean(error**2))) if len(error) and state!='X' else np.nan,
            validation_eligible=state!='X',unit=UNITS[state]))
        axes[i,j].plot(nominal.index,nominal[state],label='Modelo congelado')
        axes[i,j].scatter(support.t_h,support[col],s=12,color='black',label='Biomasa: contexto, baja confianza' if state=='X' else 'Muestras reservadas')
        draw_nut_state(axes[i,j],lab,state)
        axes[i,j].set(title=f'{lab} · {state}',xlabel='t [h]',ylabel=UNITS[state])
axes[0,0].legend(fontsize=7)
axes[0,1].legend(fontsize=7)
display_inline_figure(fig)
display(pd.DataFrame(STATE_FIT))

## 3. Qué significa perturbar y cómo se mide la información

Cada columna se obtiene cambiando **sólo un estado** y recalculando la cadena completa.
Para una escala física d_j y un paso pequeño ε:

\[J_{ij}=\frac{y_i(z+\epsilon d_j e_j)-y_i(z-\epsilon d_j e_j)}{2\epsilon\sigma_i}.\]

Si el cambio negativo viola la no negatividad, se usa diferencia hacia adelante y se registra.
J mide el efecto de **una unidad de la escala exploratoria**, no el de ε. El paso ε se reduce
a la mitad para verificar estabilidad. Las escalas se publican y no son una estimación de P₀.

Se usa SVD de J, sin formar JᵀJ para calcular rango. Un valor singular s<1 indica una dirección
con cambio agregado inferior a una unidad de ruido bajo las escalas elegidas; es un umbral
diagnóstico, **no un teorema ni un límite universal**. Se reportan rango numérico, espectro,
combinación más débil y error inferior lineal sin prior sólo cuando hay rango completo.
No se presenta la pseudoinversa de una matriz singular como certeza finita.
El rango numérico usa max(10⁻⁶,10⁻⁸·s_max) en las unidades normalizadas, para no contar redondeo
como información. También se reporta cuántos valores singulares superan el cambio espectral
observado al reducir el paso FD; este chequeo no sustituye una prueba matemática.
En límites N/O₂≈0 se usan derivadas direccionales hacia adelante. La SVD no impone el cono de
perturbaciones físicamente factibles y sus resultados allí son exploratorios.

σ=max(0.04,0.10·pico observado) g/L/h es una hipótesis heredada del ajuste, no ruido blanco medido.
La cadencia primaria es 3 h; se repite a 1 y 6 h para mostrar dependencia del muestreo.
Muestras bajo LOD se excluyen: no se tratan como ceros gaussianos. Los rangos prácticos dependen
de σ y de las escalas. Xd y Gly sirven de controles negativos: no retroalimentan esta salida.

In [ ]:
def fd_jacobian(forward,z,names,scales,extract,eps):
    baseline=extract(forward(z));columns=[];schemes=[]
    for j,name in enumerate(names):
        delta=eps*scales[name]; plus=z.copy();plus[j]+=delta
        yp=extract(forward(plus))
        if z[j]>=delta:
            minus=z.copy();minus[j]-=delta
            col=(yp-extract(forward(minus)))/(2*eps);scheme='central'
        else:
            col=(yp-baseline)/eps;scheme='forward_boundary'
        columns.append(col);schemes.append(scheme)
    return np.column_stack(columns),schemes
def spectrum(J,names):
    _,s,Vt=np.linalg.svd(J,full_matrices=True)
    s=np.pad(s,(0,len(names)-len(s)))
    threshold=max(CONFIG['rank_atol'],CONFIG['rank_rtol']*s[0]) if len(s) else CONFIG['rank_atol']
    rank=int(np.sum(s>threshold))
    practical=int(np.sum(s>=1.))
    weakest=Vt[-1] if len(Vt) else np.zeros(len(names))
    # Sin prior y con rango deficiente, la cota conjunta no se publica como finita.
    bounds=np.sqrt(np.sum((Vt.T/s)**2,axis=1)) if rank==len(names) else np.full(len(names),np.inf)
    condition=float(s[0]/s[-1]) if rank==len(names) else np.inf
    return dict(s=s,rank=rank,practical=practical,condition=condition,weakest=weakest,bounds=bounds)
def summarize(J,names,meta):
    s=spectrum(J,names)
    row={**meta,'n_rows':len(J),'n_unknowns':len(names),'rank_numeric':s['rank'],
         'directions_s_ge_1':s['practical'],'s_max':s['s'][0] if len(s['s']) else 0.,
         's_min':s['s'][-1] if len(s['s']) else 0.,'condition':s['condition']}
    if 'fd_error_spectral_norm' in meta:
        row['rank_above_fd_error']=int(np.sum(s['s']>max(meta['fd_error_spectral_norm'],CONFIG['rank_atol'])))
    direction={**meta,**dict(zip(names,s['weakest']))}
    return row,direction,s
def observation_support(lab,start,end,cadence):
    obs=co2_hourly[co2_hourly.lab.eq(lab)&co2_hourly.co2_valid&~co2_hourly.left_censored].copy()
    obs=obs[obs.t_h.between(start,end)].sort_values('t_h')
    selected=[];last=-np.inf
    for ti in obs.t_h:
        if ti-last>=cadence-1e-9:selected.append(ti);last=ti
    return np.asarray(selected,float)
SIGMA={lab:max(CONFIG['sigma_floor'],CONFIG['sigma_relative_peak']*FIT_TABLE.set_index('lab').loc[lab,'observed_peak']) for lab in LABS}
def fd_check(A,B):
    return np.linalg.norm(A-B,axis=0)/np.maximum(np.maximum(np.linalg.norm(A,axis=0),np.linalg.norm(B,axis=0)),1e-8)
SUMMARY=[];DIRECTIONS=[];FD_ROWS=[];JACOBIANS={};DETAILS=[];FD_RESOLUTION={}

## 4. Condiciones iniciales desde t₀: diagnóstico condicionado

El gate se obtiene una sola vez de la referencia nominal y se mantiene igual al perturbar estados.
Recalcular el cruce discreto G+F/E introduciría saltos de encendido y una falsa sensibilidad.
Por ello este bloque no atribuye información de un mecanismo de activación causal todavía no validado.
La normalización grid_max del transitorio se conserva; tampoco se presenta como un componente online.
Cd₀ y O₂₀ se perturban como condiciones auxiliares independientes. Xd y Gly no deben aportar información.
Una condición inicial distinguible por toda la historia no implica que el estado actual sea observable
en una ventana corta: eso se analiza en el bloque siguiente.

In [ ]:
FULL_PERTURBED={}
for lab in LABS:
    z=initial_state(recent_batches[lab],CO2_PARAMS)
    forward=lambda zz,lab=lab:full_forward(lab,zz,fixed_gate=GATES[lab])
    def extract(frame): return frame.predicted.to_numpy()
    started=time.perf_counter()
    D,schemes=fd_jacobian(forward,z,STATE_NAMES,STATE_SCALE,extract,CONFIG['fd_fraction'])
    Dhalf,_=fd_jacobian(forward,z,STATE_NAMES,STATE_SCALE,extract,CONFIG['fd_check_fraction'])
    FULL_PERTURBED[lab]=(D+Dhalf)/2.
    check=fd_check(D,Dhalf)
    for j,s in enumerate(STATE_NAMES):
        FD_ROWS.append(dict(analysis='x0',lab=lab,state=s,scheme=schemes[j],relative_change=check[j]))
    tt=NOMINAL[lab].index.to_numpy()
    windows={'early':(0.,24.),'pre_nutrition':(0.,PULSE2_H[lab]-1.),
             'post_nutrition_history':(PULSE2_H[lab]+1.,tt[-1]),'full_history':(0.,tt[-1])}
    for window,(a,b) in windows.items():
        for cadence in (1.,3.,6.):
            times=observation_support(lab,a,b,cadence)
            J=np.column_stack([np.interp(times,tt,Dhalf[:,j]) for j in range(len(STATE_NAMES))])/SIGMA[lab]
            error_matrix=np.column_stack([np.interp(times,tt,D[:,j]-Dhalf[:,j]) for j in range(7)])/SIGMA[lab]
            fd_error=float(np.linalg.norm(error_matrix,2)) if len(times) else 0.
            meta=dict(analysis='x0',lab=lab,window=window,cadence_h=cadence,start_h=a,end_h=b,
                      fd_error_spectral_norm=fd_error)
            for controls in (True,False):
                names=STATE_NAMES if controls else TARGET_NAMES
                subset=J if controls else J[:,:7]
                mode='with_negative_controls' if controls else 'states_fixed_parameters'
                row,direction,_=summarize(subset,names,{**meta,'mode':mode})
                SUMMARY.append(row);DIRECTIONS.append(direction)
                JACOBIANS[('x0',lab,window,cadence,mode)]=(times,subset,names)
    assert np.max(np.abs(Dhalf[:,7:]))<1e-4,'Controles negativos superan tolerancia numérica'
    print(lab,'sensibilidades x0 completas en',round(time.perf_counter()-started,1),'s')
display(pd.DataFrame(FD_ROWS).round(5))
display(pd.DataFrame(SUMMARY).query('cadence_h==3 and mode=="states_fixed_parameters"'))

## 5. Estados al inicio de ventanas pre-pulso: una prueba local sin gate de arranque

Se inicia después de que A_gate=1 y antes del primer pulso N. En esta región coinciden los
recorridos con/sin pulso, r=0 y M_N=1: el modelo puede reiniciarse con los estados
X,N,G,F,E,Cd,O₂ (más los dos controles negativos), sin inventar una memoria oculta.
Se estudian la primera ventana de hasta 24 h tras el gate y la última ventana de 24 h pre-pulso.
Si no hay soporte suficiente se informa. Se verifica que reiniciar reproduce la trayectoria nominal.

No se reinicia el modelo histórico post-pulso con sólo estos estados: allí la rampa usa dos
trayectorias contrafactuales upstream. Hacerlo sin ampliar la memoria cambiaría el modelo.
El análisis post-pulso del bloque anterior es de **condiciones iniciales**, no observabilidad del
estado actual para un EKF de siete estados. Esta distinción es una conclusión estructural del diagnóstico.

In [ ]:
LOCAL_CASES={};LOCAL_J={};LOCAL_SUPPORT=[]
def local_forward(lab,t,z,params=None):
    params=CO2_PARAMS if params is None else params
    v=dict(zip(STATE_NAMES,z)); original=recent_batches[lab]
    temperature=np.array([base.temperature_at(original,ti) for ti in t])
    batch=replace(original,initials={s:v[s] for s in base.STATE_NAMES},
                  pulses={channel:() for channel in base.INPUT_CHANNELS})
    core=integrate_core(batch,theta,t)
    oxygen,fferm=oxygen_general(t,core.X.to_numpy(),v['O2'],params)
    rates=co2_cross.co2_model.co2_production_g_l_h(theta,batch,core,t)
    csat=params['CO2sat_scale']*1.69*np.exp(-.032*(temperature-20.))*np.exp(.0016*core.E.to_numpy())*np.exp(-.0012*(core.G+core.F).to_numpy())
    qprod=params['bio_scale']*fferm*rates
    c,q,error=pool_general(t,csat,qprod,params['kCO2_release_h'],v['Cd'])
    result=core.copy();result['Cd']=c;result['O2']=oxygen
    result['predicted']=params['matrix_gain']*q
    result.attrs['mass_error']=error
    return result
for lab in LABS:
    n=NOMINAL[lab];tt=n.index.to_numpy()
    gate_end=float(tt[np.flatnonzero(n.gate.to_numpy()>=1.-1e-12)[0]])
    end=np.floor((PULSE2_H[lab]-1.)/GRID_H)*GRID_H
    first=np.ceil((gate_end+1.)/GRID_H)*GRID_H
    ranges={'after_gate':(first,min(first+24.,end)),
            'late_pre_nutrition':(max(first,end-24.),end)}
    for window,(start,finish) in ranges.items():
        t=tt[(tt>=start)&(tt<=finish)]
        if len(t)<2 or finish-start<6.:
            LOCAL_SUPPORT.append(dict(lab=lab,window=window,status='insufficient_duration',start_h=start,end_h=finish));continue
        z=n.loc[t[0],list(STATE_NAMES)].to_numpy(float)
        local=local_forward(lab,t,z)
        restart_error=np.max(np.abs(local.predicted.to_numpy()[1:]-n.loc[t,'predicted'].to_numpy()[1:]))
        assert restart_error<.005
        forward=lambda zz,lab=lab,t=t:local_forward(lab,t,zz)
        D,schemes=fd_jacobian(forward,z,STATE_NAMES,STATE_SCALE,lambda f:f.predicted.to_numpy(),CONFIG['fd_fraction'])
        Dhalf,_=fd_jacobian(forward,z,STATE_NAMES,STATE_SCALE,lambda f:f.predicted.to_numpy(),CONFIG['fd_check_fraction'])
        for j,s in enumerate(STATE_NAMES):
            FD_ROWS.append(dict(analysis='local',lab=lab,window=window,state=s,scheme=schemes[j],relative_change=fd_check(D,Dhalf)[j]))
        LOCAL_CASES[(lab,window)]=(t,z,local)
        LOCAL_J[(lab,window)]=Dhalf
        for cadence in (1.,3.,6.):
            times=observation_support(lab,start+GRID_H,finish,cadence)
            J=np.column_stack([np.interp(times,t,Dhalf[:,j]) for j in range(7)])/SIGMA[lab]
            error_matrix=np.column_stack([np.interp(times,t,D[:,j]-Dhalf[:,j]) for j in range(7)])/SIGMA[lab]
            fd_error=float(np.linalg.norm(error_matrix,2)) if len(times) else 0.
            FD_RESOLUTION[(lab,window,cadence)]=fd_error
            meta=dict(analysis='local',lab=lab,window=window,cadence_h=cadence,start_h=start,end_h=finish,
                      mode='states_fixed_parameters',fd_error_spectral_norm=fd_error)
            row,direction,_=summarize(J,TARGET_NAMES,meta)
            SUMMARY.append(row);DIRECTIONS.append(direction)
            JACOBIANS[('local',lab,window,cadence,'states_fixed_parameters')]=(times,J,TARGET_NAMES)
        LOCAL_SUPPORT.append(dict(lab=lab,window=window,status='evaluated',start_h=start,end_h=finish,
            restart_max_delta_g_l_h=restart_error,N_at_start=z[1],O2_at_start=z[6]))
display(pd.DataFrame(LOCAL_SUPPORT))
display(pd.DataFrame(SUMMARY).query('analysis=="local" and cadence_h==3'))

## 6. Parámetros de observación/transferencia inciertos

Para las ventanas locales se añaden perturbaciones de ganancia (10 %), k de liberación (20 %),
escala de solubilidad (20 %) y multiplicador biológico constante bio_scale (20 %).
bio_scale es una **hipótesis de amplitud incierta**, no un estado de actividad ya implementado.
El origen inicial de la ventana se conserva: sólo se cambia el parámetro a partir de ese instante.

Se calculan dos diagnósticos: J aumentado [J_estados,J_parámetros] y J_estados proyectado fuera
del espacio de las sensibilidades de parámetros. El segundo permite que los parámetros compensen
libremente cambios de estados: es un caso conservador sin prior, no un reajuste a datos.
El primero permite detectar combinaciones estado–parámetro confundidas.

In [ ]:
NUISANCE_J={}
for (lab,window),(t,z,local) in LOCAL_CASES.items():
    columns=[]
    for key in NUISANCE_NAMES:
        delta=CONFIG['fd_check_fraction']*NUISANCE_SCALE[key]
        plus={**CO2_PARAMS,key:CO2_PARAMS[key]+delta}
        minus={**CO2_PARAMS,key:CO2_PARAMS[key]-delta}
        columns.append((local_forward(lab,t,z,plus).predicted.to_numpy()-local_forward(lab,t,z,minus).predicted.to_numpy())/(2*CONFIG['fd_check_fraction']))
    NUISANCE_J[(lab,window)]=np.column_stack(columns)
    for cadence in (1.,3.,6.):
        key=('local',lab,window,cadence,'states_fixed_parameters')
        if key not in JACOBIANS:continue
        times,J,names=JACOBIANS[key]
        Jn=np.column_stack([np.interp(times,t,col) for col in NUISANCE_J[(lab,window)].T])/SIGMA[lab]
        U,s,_=np.linalg.svd(Jn,full_matrices=False)
        rank=int(np.sum(s>max(CONFIG['rank_atol'],CONFIG['rank_rtol']*s[0]))) if len(s) and s[0]>0 else 0
        Q=U[:,:rank]
        projected=J-Q@(Q.T@J)
        assert np.linalg.norm(projected)<=np.linalg.norm(J)+1e-8
        for mode,M,unknowns in (('states_nuisance_projected',projected,TARGET_NAMES),
                                ('states_and_nuisance',np.column_stack([J,Jn]),TARGET_NAMES+NUISANCE_NAMES)):
            meta=dict(analysis='local',lab=lab,window=window,cadence_h=cadence,start_h=t[0],end_h=t[-1],mode=mode,
                      fd_error_spectral_norm=FD_RESOLUTION[(lab,window,cadence)])
            row,direction,_=summarize(M,unknowns,meta)
            SUMMARY.append(row);DIRECTIONS.append(direction)
            JACOBIANS[('local',lab,window,cadence,mode)]=(times,M,unknowns)
display(pd.DataFrame(SUMMARY).query('analysis=="local" and cadence_h==3'))

## 7. ¿Qué aporta alguna medición offline?

Experimento de información, sin usar valores medidos para corregir estados: se agregan filas de
sensibilidad de X y/o N a la mitad de la ventana, como si se midieran allí con σ_X=0.06 g/L y
σ_N=0.012 g N/L (floors históricos). También se muestra X,N,G,F,E con σ_G,F=2.5 g/L y σ_E=2 g/L.
Las filas se diferencian propagando el modelo hasta ese instante: no se usa una identidad en t₀.
Estas mediciones son hipotéticas, no una validación ni la promesa de que habrá muestras allí.
Se estudia tanto con parámetros conocidos como admitiendo parámetros inciertos sin prior.

La ancla hipotética de X **no representa** las mediciones de biomasa poco confiables de LAB019–021. Es un experimento de diseño: qué pasaría si existiera una medición independiente con esa precisión. La sección 11 usa una muestra Y15 real y no asimila biomasa dinámica.

In [ ]:
ANCHOR_ROWS=[]
ANCHOR_SIGMA=dict(X=.06,N=.012,G=2.5,F=2.5,E=2.)
for (lab,window),(t,z,local) in LOCAL_CASES.items():
    key=('local',lab,window,3.,'states_fixed_parameters')
    if key not in JACOBIANS:continue
    times,J,names=JACOBIANS[key]; mid=float(t[len(t)//2])
    f=lambda zz:local_forward(lab,t,zz)
    def ex(frame):return np.array([np.interp(mid,frame.index,frame[s]) for s in ANCHOR_SIGMA])
    A,_=fd_jacobian(f,z,STATE_NAMES,STATE_SCALE,ex,CONFIG['fd_check_fraction'])
    A=A[:,:7]/np.array(list(ANCHOR_SIGMA.values()))[:,None]
    Jn=np.column_stack([np.interp(times,t,col) for col in NUISANCE_J[(lab,window)].T])/SIGMA[lab]
    An=[]
    for nuisance in NUISANCE_NAMES:
        delta=CONFIG['fd_check_fraction']*NUISANCE_SCALE[nuisance]
        p={**CO2_PARAMS,nuisance:CO2_PARAMS[nuisance]+delta};m={**CO2_PARAMS,nuisance:CO2_PARAMS[nuisance]-delta}
        An.append((ex(local_forward(lab,t,z,p))-ex(local_forward(lab,t,z,m)))/(2*CONFIG['fd_check_fraction']))
    An=np.column_stack(An)/np.array(list(ANCHOR_SIGMA.values()))[:,None]
    for label,idx in (('CO2',[]),('CO2+X',[0]),('CO2+N',[1]),('CO2+X+N',[0,1]),('CO2+X+N+G+F+E',list(range(5)))):
        fixed=np.vstack([J,A[idx]]) if idx else J
        nuisance=np.vstack([Jn,An[idx]]) if idx else Jn
        U,s,_=np.linalg.svd(nuisance,full_matrices=False)
        rank=int(np.sum(s>max(CONFIG['rank_atol'],CONFIG['rank_rtol']*s[0]))) if len(s) and s[0]>0 else 0
        projected=fixed-U[:,:rank]@(U[:,:rank].T@fixed)
        for mode,M in (('known_parameters',fixed),('nuisance_free',projected)):
            sp=spectrum(M,TARGET_NAMES)
            ANCHOR_ROWS.append(dict(lab=lab,window=window,measurements=label,mode=mode,
                sample_h=mid,rank_numeric=sp['rank'],directions_s_ge_1=sp['practical'],s_min=sp['s'][-1]))
display(pd.DataFrame(ANCHOR_ROWS))

## 8. Gráficos para interpretar el resultado

Primero se muestran las sensibilidades normalizadas y sus similitudes: columnas casi paralelas
se pueden compensar. El coseno compara forma, no amplitud; por eso se muestra también el espectro.
Los coeficientes de las direcciones débiles se expresan en las escalas publicadas, no en unidades crudas.

In [ ]:
fig,axes=plt.subplots(3,2,figsize=(14,11),layout='constrained')
for i,lab in enumerate(LABS):
    key=('local',lab,'late_pre_nutrition',3.,'states_fixed_parameters')
    if key not in JACOBIANS:continue
    times,J,names=JACOBIANS[key]
    for j,s in enumerate(names):axes[i,0].plot(times,J[:,j],label=s)
    axes[i,0].set(title=f'{lab}: efecto de una escala / sigma',xlabel='t [h]',ylabel='Sensibilidad normalizada')
    axes[i,0].legend(ncol=4,fontsize=8)
    norms=np.linalg.norm(J,axis=0);den=np.outer(norms,norms)
    cosine=np.divide(J.T@J,den,out=np.zeros_like(den),where=den>1e-12)
    im=axes[i,1].imshow(cosine,vmin=-1,vmax=1,cmap='coolwarm')
    axes[i,1].set(xticks=range(7),yticks=range(7),xticklabels=names,yticklabels=names,title='Coseno entre sensibilidades')
fig.colorbar(im,ax=axes[:,1].tolist(),shrink=.7)
display_inline_figure(fig)
fig,axes=plt.subplots(1,3,figsize=(14,4.5),layout='constrained')
for ax,lab in zip(axes,LABS):
    for mode in ('states_fixed_parameters','states_nuisance_projected'):
        key=('local',lab,'late_pre_nutrition',3.,mode)
        if key in JACOBIANS:
            s=spectrum(JACOBIANS[key][1],JACOBIANS[key][2])['s']
            ax.semilogy(range(1,len(s)+1),np.maximum(s,1e-15),'o-',label=mode)
    ax.axhline(1.,color='black',ls='--');ax.set(title=lab,xlabel='Dirección singular',ylabel='Valor singular');ax.legend(fontsize=7)
display_inline_figure(fig)
fig,axes=plt.subplots(1,3,figsize=(14,4.5),layout='constrained')
for ax,lab in zip(axes,LABS):
    key=('local',lab,'late_pre_nutrition',3.,'states_fixed_parameters')
    if key not in JACOBIANS:continue
    sp=spectrum(JACOBIANS[key][1],TARGET_NAMES)
    ax.bar(TARGET_NAMES,sp['weakest']);ax.axhline(0,color='black',lw=.7)
    ax.set(title=f'{lab}: combinación más débil',ylabel='Coeficiente en escala normalizada')
display_inline_figure(fig)

## 9. Estabilidad numérica y escenarios de robustez

Además de reducir el paso de diferenciación se repite la ventana tardía pre-pulso de LAB021 con
Δt=0.125 h, reiniciando desde el mismo estado y usando idéntica T(t). Este cambio afecta a la
discretización de O₂/pool: si cambia el espectro, el resultado no debe leerse como una propiedad
exacta de un modelo continuo. Las filas de medición y σ se conservan.
También se repite el diagnóstico x₀ de LAB020 con theta_defaults y con k, C* y ganancia ±20 %.
Son escenarios exploratorios congelados, sin optimización ni intervalos de confianza calibrados.

In [ ]:
ROBUSTNESS=[]
lab='LAB021';case=(lab,'late_pre_nutrition')
if case in LOCAL_CASES and ('local',lab,'late_pre_nutrition',3.,'states_fixed_parameters') in JACOBIANS:
    t,z,_=LOCAL_CASES[case]
    times,J,_=JACOBIANS[('local',lab,'late_pre_nutrition',3.,'states_fixed_parameters')]
    fine=np.arange(t[0],t[-1]+GRID_H/4,GRID_H/2)
    f=lambda zz:local_forward(lab,fine,zz)
    D,_=fd_jacobian(f,z,STATE_NAMES,STATE_SCALE,lambda frame:frame.predicted.to_numpy(),CONFIG['fd_check_fraction'])
    Jfine=np.column_stack([np.interp(times,fine,D[:,j]) for j in range(7)])/SIGMA[lab]
    for label,M in (('grid_0.25h',J),('grid_0.125h',Jfine)):
        row,_,_=summarize(M,TARGET_NAMES,dict(test='local_grid',scenario=label));ROBUSTNESS.append(row)
    GRID_CHANGE=fd_check(J,Jfine)
    display(pd.DataFrame({'state':TARGET_NAMES,'relative_grid_change':GRID_CHANGE}))
for label,th,pa in [('theta_full',theta,CO2_PARAMS),('theta_defaults',THETA_SETS['theta_defaults'],CO2_PARAMS),
                    ('layer_minus20',theta,{**CO2_PARAMS,**{k:.8*CO2_PARAMS[k] for k in ('matrix_gain','kCO2_release_h','CO2sat_scale')}}),
                    ('layer_plus20',theta,{**CO2_PARAMS,**{k:1.2*CO2_PARAMS[k] for k in ('matrix_gain','kCO2_release_h','CO2sat_scale')}})]:
    z=initial_state(recent_batches['LAB020'],pa)
    f=lambda zz,th=th,pa=pa:full_forward('LAB020',zz,pa,th,fixed_gate=GATES['LAB020'])
    D,_=fd_jacobian(f,z,STATE_NAMES,STATE_SCALE,lambda frame:frame.predicted.to_numpy(),CONFIG['fd_check_fraction'])
    times=observation_support('LAB020',0.,recent_time_grids['LAB020'][-1],3.)
    J=np.column_stack([np.interp(times,recent_time_grids['LAB020'],D[:,j]) for j in range(7)])/SIGMA['LAB020']
    row,_,_=summarize(J,TARGET_NAMES,dict(test='x0_scenario',scenario=label));ROBUSTNESS.append(row)
display(pd.DataFrame(ROBUSTNESS))
FD_TABLE=pd.DataFrame(FD_ROWS)
FD_TABLE['unstable_above_10pct']=FD_TABLE.relative_change>.10
display(FD_TABLE.loc[FD_TABLE.unstable_above_10pct])
print('Columnas con cambio FD >10 %:',int(FD_TABLE.unstable_above_10pct.sum()))

## 10. Lectura automática, resultados reproducibles e integridad

La conclusión se genera de las tablas de esta ejecución. Rango completo a tolerancia numérica
no implica recuperación precisa; valores singulares pequeños y dependencia de escalas/ruido
deben conservarse al interpretar el resultado. Una cota conjunta infinita indica que el diagnóstico
sin prior no permite reportar incertidumbres finitas para todos los estados a la vez.

Los CSV/NPZ/manifiesto se escriben sólo en `EKF/observability_results`. Se verifican hashes de
datos, parámetros y modelos originales. No se modifica el factor CO₂/etanol ni se recalibra theta.

In [ ]:
SUMMARY_TABLE=pd.DataFrame(SUMMARY)
DIRECTION_TABLE=pd.DataFrame(DIRECTIONS)
ANCHOR_TABLE=pd.DataFrame(ANCHOR_ROWS)
STATE_INFORMATION=[]
for key,(times,J,names) in JACOBIANS.items():
    if key[0]!='local' or key[3]!=3. or key[4]!='states_fixed_parameters':continue
    info=spectrum(J,names)
    for j,state in enumerate(names):
        STATE_INFORMATION.append(dict(lab=key[1],window=key[2],state=state,
            sensitivity_norm_over_sigma=float(np.linalg.norm(J[:,j])),
            joint_no_prior_bound_in_scale_units=info['bounds'][j],
            finite_difference_relative_change=float(FD_TABLE.loc[
                FD_TABLE.analysis.eq('local')&FD_TABLE.lab.eq(key[1])&FD_TABLE.window.eq(key[2])&FD_TABLE.state.eq(state),
                'relative_change'].iloc[0])))
display(pd.DataFrame(STATE_INFORMATION))
main=SUMMARY_TABLE.query('analysis=="local" and cadence_h==3 and mode=="states_fixed_parameters"')
for _,row in main.iterrows():
    display(Markdown(f"**{row.lab} · {row.window} ({row.start_h:g}–{row.end_h:g} h):** "
        f"rango numérico {int(row.rank_numeric)}/{int(row.n_unknowns)}; "
        f"{int(row.rank_above_fd_error)} direcciones por encima del cambio FD; "
        f"{int(row.directions_s_ge_1)} direcciones con s≥1; s mínimo={row.s_min:.3g}. "
        "La interpretación está condicionada a theta, escalas y ruido publicados."))
display(Markdown('**Controles negativos:** Xd y Gly no dejan señal en CO₂ en esta estructura. '
    '**Postnutrición:** la memoria de dos trayectorias upstream impide interpretar este diagnóstico '
    'como observabilidad de un EKF cerrado de sólo siete estados. '
    '**Alcance:** sensibilidades locales y robustez, sin ajuste a observaciones, sin EKF y sin demostración global.'))
late=SUMMARY_TABLE.query('analysis=="local" and cadence_h==3 and window=="late_pre_nutrition"')
known=late.query('mode=="states_fixed_parameters"')
free=late.query('mode=="states_nuisance_projected"')
display(Markdown(f'**Resultado principal:** con cadencia 3 h, las ventanas tardías contienen '
    f'{known.directions_s_ge_1.min():g}–{known.directions_s_ge_1.max():g} direcciones con s≥1 '
    f'para siete estados y parámetros conocidos. Al permitir compensación por parámetros inciertos, '
    f'el rango práctico es {free.directions_s_ge_1.min():g}–{free.directions_s_ge_1.max():g}. '
    'Esto describe combinaciones de estados, no un número de estados individualmente estimables. '
    'No respalda estimar simultáneamente los siete estados sin priors adicionales bajo estas hipótesis.'))
display(Markdown(f'**Calidad del modelo:** RMSE CO₂ {FIT_TABLE.RMSE_g_l_h.min():.3f}–'
    f'{FIT_TABLE.RMSE_g_l_h.max():.3f} g/L/h. Son errores de referencia, no errores de un EKF. '
    'Las comparaciones de estados anteriores muestran qué discrepancias limitan su interpretación. '
    '**Frontera O₂≈0:** si la columna cambia alrededor de 50 % al reducir el paso, el resultado '
    'corresponde a una respuesta de orden superior, no evidencia fiable de sensibilidad lineal.'))
for name,table in {'summary':SUMMARY_TABLE,'weak_directions':DIRECTION_TABLE,'finite_difference_checks':FD_TABLE,
                   'state_information':pd.DataFrame(STATE_INFORMATION),'upstream_reference_fit':pd.DataFrame(STATE_FIT),
                   'offline_anchor_information':ANCHOR_TABLE,'local_window_support':pd.DataFrame(LOCAL_SUPPORT),
                   'robustness':pd.DataFrame(ROBUSTNESS),'reference_checks':pd.DataFrame(REFERENCE_CHECK),
                   'reference_fit':FIT_TABLE,'initial_conditions':pd.DataFrame(recent_initial_rows)}.items():
    table.to_csv(OUT/f'{name}.csv',index=False)
for lab,frame in NOMINAL.items():frame.rename_axis('t_h').to_csv(OUT/f'nominal_{lab}.csv')
arrays={}
for i,(key,(times,J,names)) in enumerate(JACOBIANS.items()):
    arrays[f'J_{i}']=J;arrays[f't_{i}']=times
np.savez_compressed(OUT/'sensitivity_matrices.npz',**arrays)
assert HASH_BEFORE=={str(p.relative_to(ROOT)):sha256(p) for p in PROTECTED}
manifest=dict(config=CONFIG,state_scales=STATE_SCALE,nuisance_scales=NUISANCE_SCALE,
    sigma_by_lab=SIGMA,theta=theta,co2_parameters=CO2_PARAMS,input_sha256=HASH_BEFORE,
    matrix_index={str(i):{'key':list(key),'states':list(value[2])} for i,(key,value) in enumerate(JACOBIANS.items())},
    scientific_scope='offline local sensitivity, not formal nonlinear/global observability or EKF validation',
    python=sys.executable,protected_files=len(PROTECTED),optimizer_calls=0)
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print('VERIFICADO: datos/parámetros/modelos intactos; balance del pool y saltos N; controles negativos; optimizadores=0.')
print('Resultados:',OUT)

## 11. Incertidumbre inicial y comparación de los dos theta

### Qué pregunta responde esta ampliación

Con las mediciones iniciales, la biomasa poco confiable y parámetros conocidos con cierta incertidumbre, **¿cuánto puede reducir el CO₂ la incertidumbre de cada estado? ¿Qué añade una muestra posterior del Y15? ¿Cambia la respuesta entre `theta_full` y el theta incompleto completado con defaults?**

No demostramos observabilidad global ni ejecutamos un EKF. Es un diagnóstico condicionado al modelo actual. Se comparan LAB019–021 individualmente, sin compartir artificialmente información de parámetros entre lotes. CO₂ y química Y15 son soportes confiables; la biomasa dinámica queda fuera de la validación. Los lotes anteriores no intervienen como referencia de caudal.

Los dos casos usan exactamente `PARAM_SETS['antiguos']` del notebook 02: nueve parámetros de la calibración natural, A=0.30, subida=4 h, caída=12 h y normalización `grid_max`. La estructura sigue siendo la simplificada descrita arriba (transitorio en la producción, sin término respiratorio). Cambiar theta **no** cambia la estructura. El gate se fija al nominal de cada theta: no se atribuye información a un gate que se desplaza con una perturbación. Por ello el diagnóstico está condicionado al gate y no valida su predicción online.

### Lo que conocemos y lo que suponemos

* Las medianas iniciales de química son las mediciones m1. La referencia de X conserva Oculyze crudo × 0.03 para mantener trazabilidad, **sin suponer que sea precisa**: CV=100 % en todos los escenarios.
* No hay medición de CO₂ disuelto ni O₂ en estos archivos. Para explorar el reservorio se usa una **hipótesis**, Cd₀ mediano=0.10 g/L, CV=100 %. Se compara por separado con Cd₀=0 y 0.30. No se presenta como medición del Carbodoseur. O₂₀ se centra en el valor calculado por los parámetros antiguos, CV=50 %.
* Las incertidumbres de Y15, Anton Paar y parámetros son **supuestos exploratorios**, no especificaciones instrumentales. Los priors lognormales garantizan positividad; los valores nominales son medianas. La tabla muestra desviaciones logarítmicas y desviaciones físicas exactas del prior.
* Se perturban G y G+F, y se obtiene F=(G+F)−G. Así no se cuentan G, F y G+F como tres mediciones independientes. Se supone independencia de los canales G y G+F; la dependencia G–F se propaga. Hace falta la covarianza analítica real para refinar ese supuesto.
* Se consideran los 17 parámetros cinéticos inciertos también en `theta_defaults`: default **no significa conocido exactamente**. El FIM histórico de 11 parámetros no es una covarianza conjunta validada de los 17 más la capa gaseosa. Se supone independencia previa entre parámetros y se publica este supuesto.
* Xd₀ se fija al valor histórico y no retroalimenta la señal; Gly₀ sí tiene incertidumbre para comparar con Y15 aunque no retroalimente CO₂. `bio_scale` queda fijo en 1; no se añade como parámetro ficticio. Gate, dosis, temperatura y conversiones permanecen fijos, por lo que sus errores no están incluidos.
* Se acelera la evaluación de producción gaseosa en la misma malla usando las ecuaciones oficiales sin interpolaciones redundantes. Se verifica su igualdad con la función original a 10⁻¹² g/L/h. No se cambia la ecuación cinética. Las derivadas de γG0 y γF0 respecto a estados distintos de Gly se fijan al cero analítico: Gly no retroalimenta ninguna ecuación.

**Comparación de theta y origen de la calibración:** `run_co2_matrix_cross_validation_2026.py` carga `estimability_historical_natural/theta.csv` mediante `_load_theta` para la matriz natural (líneas 65 y 588). Es decir, la calibración antigua de la capa gaseosa se construyó con el theta completado con defaults. Un mejor RMSE de `theta_defaults` bajo esa misma capa mide compatibilidad con esa calibración, y no demuestra superioridad física frente a `theta_full`. Aquí se conserva deliberadamente la capa antigua en ambos casos, según el alcance solicitado.

In [ ]:
import ast
from scipy.linalg import cho_factor, cho_solve
from scipy.special import logsumexp
from scipy.stats import qmc, norm

# Verificación directa de los literales de PARAM_SETS['antiguos'] del SOURCE 02.
source02=nb02=json.loads((LAB2026/'notebooks/co2_reformulation/02_nutrition_structure_comparison.source.ipynb').read_text(encoding='utf-8'))
old_literal=None
for cell in source02['cells']:
    if cell['cell_type']!='code': continue
    if 'PARAM_SETS =' not in ''.join(cell['source']): continue
    for node in ast.parse(''.join(cell['source'])).body:
        if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='PARAM_SETS' for t in node.targets):
            old_literal=next(v for k,v in zip(node.value.keys,node.value.values) if ast.literal_eval(k)=='antiguos')
assert old_literal is not None
old_extras={ast.literal_eval(k):ast.literal_eval(v) for k,v in zip(old_literal.keys,old_literal.values) if k is not None}
assert old_extras==dict(A=.30,tau_rise_h=4.,tau_decay_h=12.,kernel_normalization='grid_max')
assert [v.id for k,v in zip(old_literal.keys,old_literal.values) if k is None]==['old_co2']
assert all(CO2_PARAMS[k]==v for k,v in old_extras.items())
layer_old=pd.read_csv(LAB2026/'results/co2_matrix_cross_validation_2026/fit_parameters.csv').query('calibration_matrix=="natural"')
assert all(CO2_PARAMS[r.parameter]==float(r.estimate) for r in layer_old.itertuples())
LAYER_UNCERTAIN=('kCO2_release_h','CO2sat_scale','matrix_gain','O2_qmax_mg_gdw_h',
                 'pulse_t_rise_h','pulse_activity_gain','A','tau_rise_h','tau_decay_h')
INITIAL_CHANNELS=('X','N','G','GF','E','Gly','Cd','O2')
ETA_NAMES=tuple('initial:'+s for s in INITIAL_CHANNELS)+tuple('theta:'+s for s in base.FULL17)+tuple('co2:'+s for s in LAYER_UNCERTAIN)
U_STATES=('X','N','G','F','E','Cd','O2','Gly')
U_OUTPUTS=(*U_STATES,'predicted')
PRIOR_SCENARIOS={'confianza_alta':dict(chem=.5,theta_cv=.10,layer_cv=.20),
                 'referencia':dict(chem=1.,theta_cv=.25,layer_cv=.50),
                 'confianza_baja':dict(chem=2.,theta_cv=.50,layer_cv=1.)}
CHEM_SIGMA=dict(N=.010,G=1.,GF=2.,E=.5,Gly=.10)  # unidades físicas, supuestos
U_CONFIG=dict(fd_log_step=.002,mc_power=7,seed=20261013,initial_Cd_median=.10,
              covariance_method='whitened prior + Cholesky',gas_cadence_h=3.,
              ESS_min=20,ESS_fraction_min=.15,gas_noise='SIGMA; stress x2',gate='fixed_per_theta')
def center_initial(lab):
    z=dict(zip(STATE_NAMES,initial_state(recent_batches[lab],CO2_PARAMS)))
    return {**z,'GF':z['G']+z['F'],'Cd':U_CONFIG['initial_Cd_median']}
def log_sd_cv(cv): return np.sqrt(np.log1p(cv**2))
def prior_sd(lab,scenario):
    center=center_initial(lab); cfg=PRIOR_SCENARIOS[scenario]; out=[]
    for s in INITIAL_CHANNELS:
        cv=(1. if s in ('X','Cd') else .50 if s=='O2' else cfg['chem']*CHEM_SIGMA[s]/center[s])
        out.append(log_sd_cv(cv))
    out.extend([log_sd_cv(cfg['theta_cv'])]*len(base.FULL17))
    out.extend([log_sd_cv(cfg['layer_cv'])]*len(LAYER_UNCERTAIN))
    return np.array(out)
def uncertainty_forward(lab,theta_name,eta=None,gate=None):
    eta=np.zeros(len(ETA_NAMES)) if eta is None else np.asarray(eta)
    ini=center_initial(lab); th=THETA_SETS[theta_name].copy(); params=CO2_PARAMS.copy()
    for name,d in zip(ETA_NAMES,eta):
        group,key=name.split(':'); target={'initial':ini,'theta':th,'co2':params}[group]
        target[key]*=np.exp(d)
    ini['F']=ini.pop('GF')-ini['G']
    if ini['F']<0: raise ValueError('draw G > G+F; composición imposible')
    return full_forward(lab,z=np.array([ini[s] for s in STATE_NAMES]),params=params,
                        theta_override=th,fixed_gate=gate)
PRIOR_ROWS=[]
for theta_name in THETA_SETS:
    for lab in LABS:
        for scenario in PRIOR_SCENARIOS:
            sd=prior_sd(lab,scenario); centers={**{'initial:'+s:v for s,v in center_initial(lab).items()},
                **{'theta:'+s:v for s,v in THETA_SETS[theta_name].items()},**{'co2:'+s:CO2_PARAMS[s] for s in LAYER_UNCERTAIN}}
            for name,sig in zip(ETA_NAMES,sd):
                median=centers[name]
                physical_sd=median*np.exp(.5*sig**2)*np.sqrt(np.expm1(sig**2))
                PRIOR_ROWS.append(dict(theta=theta_name,lab=lab,scenario=scenario,unknown=name,
                                       median=median,sd_log=sig,sd_physical=physical_sd))
PRIOR_TABLE=pd.DataFrame(PRIOR_ROWS)
display(PRIOR_TABLE.query('theta=="theta_full" and lab=="LAB019" and scenario=="referencia"'))
display(THETA_TABLE)
print('Comparación controlada:',len(ETA_NAMES),'incógnitas por lote; mismos parámetros antiguos en ambos theta.')


MODEL_FUNCTION_NAMES = ['anchor_rows', 'at_time', 'bracket_from_core', 'center_initial', 'cumulative_clock', 'curve', 'display_inline_figure', 'ex', 'extract', 'fd_check', 'fd_jacobian', 'full_forward', 'initial_state', 'integrate_core', 'load_offline_samples', 'load_parameters', 'load_y15', 'local_condition', 'local_forward', 'log_sd_cv', 'make_profile', 'measurement_sigma', 'numeric', 'observation_support', 'oculyze_m1_biomass', 'oxygen_fraction', 'oxygen_general', 'pool', 'pool_general', 'prior_sd', 'production_on_core_grid', 'pulse_multiplier', 'read_csv_window', 'sha256', 'simulate_clock', 'spectrum', 'speed_factor', 'summarize', 'uncertainty_forward']
def model_function_code_signature():
    source = LAB2026/'notebooks/EKF/co2_local_observability.source.ipynb'
    document=json.loads(source.read_text(encoding='utf-8'));nodes=[]
    for cell in document['cells']:
        if cell['cell_type']!='code':continue
        for node in ast.walk(ast.parse(''.join(cell['source']))):
            if isinstance(node,(ast.FunctionDef,ast.AsyncFunctionDef)) and node.name in MODEL_FUNCTION_NAMES:
                nodes.append((node.name,ast.dump(node,include_attributes=False)))
    return hashlib.sha256(json.dumps(sorted(nodes),ensure_ascii=True).encode()).hexdigest()
FORWARD_CODE_SHA256 = model_function_code_signature()
LEGACY_FORWARD_CODE_SHA256 = '2327830ea6a0450d4858fc346529a71c187f2d0307a5a4791c31c4aa7dbba61a'

REUSE_VALIDATED_TRAJECTORIES = True
SIMULATION_CACHE = {}
cache_files = [OUT/'uncertainty_manifest.json',OUT/'uncertainty_arrays.npz',
               OUT/'uncertainty_FD_checks.csv',OUT/'uncertainty_MC_status.csv',
               OUT/'uncertainty_Y15_anchor.csv']
if REUSE_VALIDATED_TRAJECTORIES and all(p.exists() for p in cache_files):
    cm=json.loads(cache_files[0].read_text(encoding='utf-8'))
    matches=(cm['input_sha256']==HASH_BEFORE and cm['config']==U_CONFIG
             and cm['scenarios']==PRIOR_SCENARIOS and cm['chem_sigma']==CHEM_SIGMA
             and cm['theta_sets']==THETA_SETS and cm['co2_parameters']==CO2_PARAMS
             and cm['unknowns']==list(ETA_NAMES) and cm['outputs']==list(U_OUTPUTS)
             and cm.get('forward_code_sha256',LEGACY_FORWARD_CODE_SHA256)==FORWARD_CODE_SHA256)
    if matches:
        with np.load(cache_files[1]) as saved: cached_arrays={k:saved[k] for k in saved.files}
        cached_fd=pd.read_csv(cache_files[2]); cached_status=pd.read_csv(cache_files[3])
        cached_anchor=pd.read_csv(cache_files[4]).set_index('lab')
        for idx,key in cm['case_index'].items():
            key=(key['theta'],key['lab']);i=int(idx)
            SIMULATION_CACHE[key]=dict(D=cached_arrays[f'D_{i}'],Dhalf=cached_arrays[f'Dhalf_{i}'],
                t=cached_arrays[f't_{i}'],ensemble=cached_arrays[f'mc_draws_{i}'],
                accepted=cached_arrays[f'accepted_draws_{i}'],draws=cached_arrays['standard_normal_draws'],
                nominal=pd.read_csv(OUT/f'uncertainty_nominal_{key[0]}_{key[1]}.csv',index_col='t_h'),
                FD=cached_fd[cached_fd.theta.eq(key[0])&cached_fd.lab.eq(key[1])],
                status=cached_status[cached_status.theta.eq(key[0])&cached_status.lab.eq(key[1])],
                anchor=cached_anchor.loc[key[1]])
print('Caché de simulaciones con entradas verificadas:',len(SIMULATION_CACHE),'casos; observaciones NUT se procesan de nuevo.')


### 11.1 Propagar desde t₀ y combinar la información sin perder correlaciones

Sea η el cambio logarítmico de las cantidades inciertas y L la matriz diagonal de sus desviaciones previas. Escribimos η=L u, con u de covarianza identidad. Simulamos **todo el historial desde t₀**, incluidos los pulsos y las dos trayectorias auxiliares de nutrición. B(t)=∂x(t)/∂η y J=∂y/∂η se calculan por diferencias centrales. No copiamos la incertidumbre inicial a las 50 h ni reiniciamos un modelo de siete estados después del pulso.

Para cada instante sólo se usan observaciones de CO₂ anteriores o iguales a ese instante, separadas al menos 3 h, válidas y sobre el límite de detección. Los ceros censurados no se tratan como gaussianos exactos. Su exclusión deja el análisis condicionado a ese soporte; no es la verosimilitud completa de datos censurados. Se supone ruido independiente; 6 h y ruido doble son controles frente a esa hipótesis.

Con H=R⁻¹ᐟ² J L, la covarianza local queda Cᵤ=(I+HᵀH)⁻¹ y la incertidumbre de estados es B L Cᵤ Lᵀ Bᵀ. Usamos Cholesky y verificamos que añadir información no aumente la covarianza. La disminución de σ mide **información potencial local**. No basta para demostrar que la estimación es correcta.

La muestra ancla es la primera muestra real con Y15 completo a partir de 24 h, elegida por tiempo y calidad, antes de mirar los residuos. Se comparan: prior solo, prior+CO₂, prior+CO₂+YAN de esa muestra y prior+CO₂+Y15 completo (N,G,G+F,Gly). La misma muestra se excluye completamente de la validación posterior, incluso para canales no asimilados. E de Anton Paar se reserva para contraste. No se asimilan las muestras iniciales otra vez.

**Escala de ruido:** SIGMA se fija como el máximo entre 0.04 g/L/h y el 10 % del pico observado de cada lote. Es una escala retrospectiva para este diagnóstico, no una estimación disponible online desde t₀. La afirmación de usar sólo observaciones anteriores a cada instante se refiere a las filas asimiladas, y presupone conocido ese ruido. La validación operativa requerirá calibrarlo de forma independiente.

**Alcance de las covarianzas:** se propagan incertidumbres iniciales y de parámetros mediante un modelo determinista. No se añade ruido de proceso Q ni perturbaciones no modeladas durante la fermentación. Estas covarianzas no son las P de un EKF operativo; omitir esas fuentes puede subestimar la incertidumbre real.

La opción `REUSE_VALIDATED_TRAJECTORIES` reutiliza sensibilidades y simulaciones no
lineales ya verificadas cuando coinciden hashes de entradas/modelos, configuraciones,
parámetros, código de las funciones, mallas, nominales y muestras ancla. Añadir NUT sólo cambia el contraste
químico: no modifica los drivers ni iniciales. Las comparaciones con las nuevas
muestras y sus figuras se recalculan. Si el control falla, se simula de nuevo.


In [ ]:
U_CASES={}; U_FD=[]; U_POINT_FIT=[]; SENTINELS={}
for lab in LABS:
    candidates=chem[chem.lab.eq(lab)&~chem.is_initial&~chem.soft_flag_GF&chem.t_h.ge(24.)]
    candidates=candidates.dropna(subset=['N_kg_m3','G_g_l','GF_g_l','Gly_g_l']).sort_values('t_h')
    assert len(candidates),'No hay muestra Y15 completa posterior a 24 h'
    SENTINELS[lab]=candidates.iloc[0]
for theta_name in THETA_SETS:
    for lab in LABS:
        started=time.perf_counter(); nominal=uncertainty_forward(lab,theta_name)
        gate=(nominal.index.to_numpy(),nominal.gate.to_numpy())
        t=nominal.index.to_numpy(); f0=nominal[list(U_OUTPUTS)].to_numpy()
        cached=SIMULATION_CACHE.get((theta_name,lab))
        if cached is not None:
            anchor=SENTINELS[lab]
            same_anchor=(str(cached['anchor']['sample'])==str(anchor['sample'])
                         and np.isclose(cached['anchor'].t_h,anchor.t_h))
            for col in ('N_kg_m3','G_g_l','GF_g_l','Gly_g_l'):
                same_anchor &= np.isclose(float(cached['anchor'][col]),float(anchor[col]),rtol=1e-12)
            cached_nominal=cached['nominal']
            valid=(np.array_equal(t,cached['t']) and same_anchor
                   and list(cached_nominal.columns)==list(nominal.columns)
                   and np.allclose(cached_nominal.to_numpy(),nominal.to_numpy(),rtol=1e-10,atol=1e-11))
            if not valid: cached=None; SIMULATION_CACHE.pop((theta_name,lab),None)
        if cached is not None:
            D=cached['D'];Dhalf=cached['Dhalf']
            U_FD.extend(cached['FD'].to_dict('records'))
            print(theta_name,lab,'sensibilidades reutilizadas: inputs, malla, nominal y ancla verificados',flush=True)
        else:
            derivatives=[];derivatives_half=[]; eps=U_CONFIG['fd_log_step']
            for j,name in enumerate(ETA_NAMES):
                delta=np.zeros(len(ETA_NAMES)); delta[j]=eps
                plus=uncertainty_forward(lab,theta_name,delta,gate)
                minus=uncertainty_forward(lab,theta_name,-delta,gate)
                derivative=(plus[list(U_OUTPUTS)].to_numpy()-minus[list(U_OUTPUTS)].to_numpy())/(2*eps)
                # Todos los canales: reducir el paso comprueba robustez numérica de información local.
                plus2=uncertainty_forward(lab,theta_name,delta/2,gate)
                minus2=uncertainty_forward(lab,theta_name,-delta/2,gate)
                d2=(plus2[list(U_OUTPUTS)].to_numpy()-minus2[list(U_OUTPUTS)].to_numpy())/eps
                raw_relative=np.linalg.norm(d2-derivative)/max(np.linalg.norm(d2),np.linalg.norm(derivative),1e-9)
                if name in ('theta:gammaG0','theta:gammaF0','initial:Gly'):
                    independent=[k for k,s in enumerate(U_OUTPUTS) if s!='Gly']
                    derivative[:,independent]=0.;d2[:,independent]=0.
                relative=np.linalg.norm(d2-derivative)/max(np.linalg.norm(d2),np.linalg.norm(derivative),1e-9)
                derivatives.append(derivative);derivatives_half.append(d2)
                U_FD.append(dict(theta=theta_name,lab=lab,unknown=name,relative_change=relative,raw_relative_change=raw_relative))
            D=np.stack(derivatives,axis=-1)
            Dhalf=np.stack(derivatives_half,axis=-1)
        times=observation_support(lab,0.,t[-1],3.)
        observed=co2_hourly[co2_hourly.lab.eq(lab)].set_index('t_h').co2_observed_g_l_h
        y=observed.reindex(times).to_numpy()
        J=np.vstack([np.interp(times,t,D[:,-1,j]) for j in range(len(ETA_NAMES))]).T
        pred=np.interp(times,t,nominal.predicted)
        zero=full_forward(lab,theta_override=THETA_SETS[theta_name],fixed_gate=gate)
        z30=initial_state(recent_batches[lab],CO2_PARAMS);z30[STATE_NAMES.index('Cd')]=.30
        high=full_forward(lab,z=z30,theta_override=THETA_SETS[theta_name],fixed_gate=gate)
        U_POINT_FIT.append(dict(theta=theta_name,lab=lab,n_CO2=len(y),
            RMSE_CO2=float(np.sqrt(np.mean((pred-y)**2))),sigma_CO2=SIGMA[lab],
            RMSE_over_sigma=float(np.sqrt(np.mean((pred-y)**2)))/SIGMA[lab],
            Cd0_effect_zero_vs_030=float(np.max(np.abs(np.interp(times,t,high.predicted-zero.predicted))))))
        Jhalf=np.vstack([np.interp(times,t,Dhalf[:,-1,j]) for j in range(len(ETA_NAMES))]).T
        U_CASES[theta_name,lab]=dict(nominal=nominal,t=t,D=D,Dhalf=Dhalf,times=times,y=y,J=J,Jhalf=Jhalf,pred=pred,gate=gate)
        checkpoint={}
        for ii,cc in enumerate(U_CASES.values()):
            for kk in ('t','D','Dhalf','times','y','J','Jhalf','pred'):checkpoint[f'{kk}_{ii}']=cc[kk]
            checkpoint[f'nominal_{ii}']=cc['nominal'].to_numpy()
        np.savez_compressed(OUT/'uncertainty_case_checkpoint.npz',**checkpoint)
        (OUT/'uncertainty_case_checkpoint.json').write_text(json.dumps(dict(
            cases=[list(key) for key in U_CASES],columns=list(nominal.columns),FD=U_FD,point_fit=U_POINT_FIT,
            input_sha256=HASH_BEFORE)),encoding='utf-8')
        print(theta_name,lab,'sensibilidades completas y paso/2:',round(time.perf_counter()-started,1),'s',flush=True)
display(pd.DataFrame([dict(lab=lab,sample=str(s['sample']),t_h=float(s.t_h)) for lab,s in SENTINELS.items()]))
display(pd.DataFrame(U_POINT_FIT).round(4))
display(pd.DataFrame(U_FD).groupby(['theta','lab']).relative_change.agg(['max','median']))
display(pd.DataFrame(U_FD).query('relative_change>.05'))
assert np.isfinite([r['relative_change'] for r in U_FD]).all()

In [ ]:
MODES=('prior','CO2','CO2_YAN','CO2_Y15')
Y15_CHANNELS=('N','G','GF','Gly')
Y15_COLS=dict(N='N_kg_m3',G='G_g_l',GF='GF_g_l',Gly='Gly_g_l')
def at_time(case,ti):
    B=np.vstack([np.array([np.interp(ti,case['t'],case['D'][:,k,j]) for j in range(len(ETA_NAMES))]) for k in range(len(U_STATES))])
    x=np.array([np.interp(ti,case['t'],case['nominal'][s]) for s in U_STATES])
    return x,B
def measurement_sigma(channel,value,scenario='referencia'):
    return PRIOR_SCENARIOS[scenario]['chem']*max(CHEM_SIGMA[channel],.02*abs(value))
def anchor_rows(case,lab,scenario,channels):
    sentinel=SENTINELS[lab]; x,B=at_time(case,float(sentinel.t_h)); rows=[];residual=[]
    for s in channels:
        if s=='GF': deriv=B[U_STATES.index('G')]+B[U_STATES.index('F')]; mean=x[U_STATES.index('G')]+x[U_STATES.index('F')]
        else: deriv=B[U_STATES.index(s)];mean=x[U_STATES.index(s)]
        observed=float(sentinel[Y15_COLS[s]]); sig=measurement_sigma(s,observed,scenario)
        rows.append(deriv/sig); residual.append((observed-mean)/sig)
    return np.asarray(rows),np.asarray(residual)
def local_condition(case,lab,scenario,ti,mode,cadence=3.,noise_factor=1.,gas_bias_cv=0.):
    sd=prior_sd(lab,scenario); selected=case['times']<=ti+1e-9
    if cadence==6.:
        chosen=observation_support(lab,0.,min(ti,case['t'][-1]),6.)
        selected &= np.isin(case['times'],chosen)
    J=case['J'][selected]/(SIGMA[lab]*noise_factor)
    r=(case['y'][selected]-case['pred'][selected])/(SIGMA[lab]*noise_factor)
    if mode=='prior': J=np.empty((0,len(sd)));r=np.empty(0)
    if mode in ('CO2_YAN','CO2_Y15') and ti>=float(SENTINELS[lab].t_h):
        Ja,ra=anchor_rows(case,lab,scenario,('N',) if mode=='CO2_YAN' else Y15_CHANNELS)
        J=np.vstack([J,Ja]);r=np.r_[r,ra]
    H=J*sd
    # Constant unknown gas bias is integrated as an additional nuisance coordinate in stress tests.
    if gas_bias_cv and len(J):
        gas_rows=int(np.sum(selected)) if mode!='prior' else 0
        bias=np.zeros(len(J));bias[:gas_rows]=gas_bias_cv*case['pred'][selected]/(SIGMA[lab]*noise_factor)
        H=np.column_stack([H,bias])
    factor=cho_factor(np.eye(H.shape[1])+H.T@H,lower=True,check_finite=False)
    C=cho_solve(factor,np.eye(H.shape[1]),check_finite=False)
    mean=cho_solve(factor,H.T@r,check_finite=False)
    assert np.linalg.eigvalsh(C).min()>-1e-10 and np.linalg.eigvalsh(np.eye(len(C))-C).min()>-1e-8
    return C[:len(sd),:len(sd)],mean[:len(sd)],sd,H
U_INTERVALS=[];U_WEAK=[];U_CORR=[];U_POST={};U_VALIDATION=[];U_STRESS=[]
for (theta_name,lab),case in U_CASES.items():
    sample_times=chem.loc[chem.lab.eq(lab)&~chem.is_initial,'t_h'].to_numpy(float)
    checkpoints=np.unique(np.r_[0.,24.,48.,72.,96.,120.,144.,case['t'][-1],sample_times])
    checkpoints=checkpoints[(checkpoints>=0)&(checkpoints<=case['t'][-1])]
    for scenario in PRIOR_SCENARIOS:
        for ti in checkpoints:
            x,B=at_time(case,ti)
            for mode in MODES:
                C,mean,sd,H=local_condition(case,lab,scenario,ti,mode)
                BL=B*sd; sigma=np.sqrt(np.maximum(np.einsum('ij,jk,ik->i',BL,C,BL),0.))
                prior_sigma=np.linalg.norm(BL,axis=1)
                shifted=x+BL@mean
                for k,s in enumerate(U_STATES):
                    U_INTERVALS.append(dict(theta=theta_name,lab=lab,scenario=scenario,t_h=ti,mode=mode,state=s,
                        nominal=x[k],local_mean=shifted[k],sigma=sigma[k],sigma_prior=prior_sigma[k],
                        sigma_ratio=sigma[k]/prior_sigma[k] if prior_sigma[k]>1e-12 else np.nan,
                        lower95=shifted[k]-1.96*sigma[k],upper95=shifted[k]+1.96*sigma[k],
                        max_shift_prior_sd=float(np.max(np.abs(mean)))))
        for mode in MODES[1:]:
            C,mean,sd,H=local_condition(case,lab,scenario,case['t'][-1],mode)
            U_POST[theta_name,lab,scenario,mode]=(C,mean,sd)
            vals,vecs=np.linalg.eigh(C);weak=vecs[:,-1]
            if weak[np.argmax(np.abs(weak))]<0:weak=-weak
            U_WEAK.append(dict(theta=theta_name,lab=lab,scenario=scenario,mode=mode,
                variance_remaining=float(vals[-1]),information_directions_gt1=int(np.sum(np.linalg.svd(H,compute_uv=False)>1.)),
                **dict(zip(ETA_NAMES,weak))))
            corr=C/np.sqrt(np.outer(np.diag(C),np.diag(C)))
            pairs=sorted(((abs(corr[i,j]),i,j) for i in range(len(sd)) for j in range(i)),reverse=True)[:8]
            for _,i,j in pairs:U_CORR.append(dict(theta=theta_name,lab=lab,scenario=scenario,mode=mode,
                                               a=ETA_NAMES[i],b=ETA_NAMES[j],correlation=corr[i,j]))
    for cadence,noise,bias in ((3.,1.,0.),(6.,1.,0.),(3.,2.,0.),(3.,1.,.20)):
        C,mean,sd,H=local_condition(case,lab,'referencia',case['t'][-1],'CO2',cadence,noise,bias)
        x,B=at_time(case,case['t'][-1]); BL=B*sd
        for k,s in enumerate(U_STATES):
            U_STRESS.append(dict(theta=theta_name,lab=lab,cadence_h=cadence,noise_factor=noise,
                gas_constant_gain_nuisance_cv=bias,state=s,
                sigma_ratio=np.sqrt(max(0.,BL[k]@C@BL[k]))/max(np.linalg.norm(BL[k]),1e-12)))
    # Reservar toda la muestra ancla y todas las mediciones iniciales; no validar biomasa.
    held=chem[chem.lab.eq(lab)&~chem.is_initial&~chem.is_NUT&~chem['sample'].astype(str).eq(str(SENTINELS[lab]['sample']))]
    for sample in held.itertuples():
        ti=float(sample.t_h)
        if not 0<=ti<=case['t'][-1]:continue
        for mode in MODES[1:]:
            C,mean,sd,H=local_condition(case,lab,'referencia',ti,mode)
            x,B=at_time(case,ti);BL=B*sd
            for s,col in {**Y15_COLS,'E':'E_g_l'}.items():
                if s in ('G','GF') and sample.soft_flag_GF:continue
                measured=getattr(sample,col)
                if not np.isfinite(measured):continue
                if s=='GF': b=BL[U_STATES.index('G')]+BL[U_STATES.index('F')];reference=x[U_STATES.index('G')]+x[U_STATES.index('F')]
                else:b=BL[U_STATES.index(s)];reference=x[U_STATES.index(s)]
                pred=reference+b@mean; sig=np.sqrt(max(b@C@b,0.)+measurement_sigma(s,measured)**2)
                U_VALIDATION.append(dict(theta=theta_name,lab=lab,mode=mode,sample=str(sample.sample),t_h=ti,
                    state=s,observed=measured,nominal=reference,local_mean=pred,total_sigma=sig,
                    standardized_residual=(pred-measured)/sig,inside_local95=abs(pred-measured)<=1.96*sig,
                    max_shift_prior_sd=float(np.max(np.abs(mean)))))
U_INTERVAL_TABLE=pd.DataFrame(U_INTERVALS);U_VALID_TABLE=pd.DataFrame(U_VALIDATION)
U_FD_IMPACT=[]
for (name,lab),case in U_CASES.items():
    alternative={**case,'D':case['Dhalf'],'J':case['Jhalf']}
    for ti in (96.,case['t'][-1]):
        for mode in ('CO2','CO2_Y15'):
            C,mu,sd,H=local_condition(case,lab,'referencia',ti,mode)
            C2,mu2,sd,H2=local_condition(alternative,lab,'referencia',ti,mode)
            _,B=at_time(case,ti);_,B2=at_time(alternative,ti);BL=B*sd;BL2=B2*sd
            sig=np.sqrt(np.maximum(np.einsum('ij,jk,ik->i',BL,C,BL),0.))
            sig2=np.sqrt(np.maximum(np.einsum('ij,jk,ik->i',BL2,C2,BL2),0.))
            prior=np.linalg.norm(BL,axis=1)
            for k,s in enumerate(U_STATES):U_FD_IMPACT.append(dict(theta=name,lab=lab,t_h=ti,mode=mode,state=s,
                sigma=sig[k],sigma_half=sig2[k],absolute_change=abs(sig2[k]-sig[k]),
                change_over_prior=abs(sig2[k]-sig[k])/max(prior[k],1e-10)))
display(pd.DataFrame(U_FD_IMPACT).groupby(['theta','lab']).agg(max_absolute_sigma_change=('absolute_change','max'),max_change_over_prior=('change_over_prior','max')))
assert not U_VALID_TABLE.state.eq('X').any()
assert all(not U_VALID_TABLE[U_VALID_TABLE.lab.eq(lab)]['sample'].eq(str(SENTINELS[lab]['sample'])).any() for lab in LABS)
display(U_INTERVAL_TABLE.query('scenario=="referencia" and state in ["X","N","G","F","E"]')
        .groupby(['theta','lab','mode']).sigma_ratio.median().unstack('mode').round(3))
display(pd.DataFrame(U_CORR).query('scenario=="referencia" and mode=="CO2"').groupby(['theta','lab']).head(3))

### 11.2 Comprobación no lineal y contraste independiente

La actualización anterior también permite calcular una media **local lineal**. Si los datos exigen desplazamientos grandes, puede salir del dominio donde la linealización tiene sentido. No interpretamos automáticamente sus bandas como intervalos creíbles ni una cobertura aparente como validación del EKF.

Se realizan 128 simulaciones no lineales por combinación lote/theta con el prior de referencia, usando los mismos puntos Sobol en las seis combinaciones. Se registran las simulaciones que incumplen no negatividad, integración o balance. Las bandas del conjunto exitoso están condicionadas a esa admisibilidad: **si falla más del 10 %, no representan el prior completo**. No se recortan trayectorias negativas para hacerlas pasar como válidas.

Se comparan cuantiles y desviaciones del conjunto admisible con la propagación local. Se intenta reponderarlo por CO₂ y CO₂+Y15; ESS comprueba cuántas simulaciones sobreviven. **Si ESS<max(20,15 % del conjunto), o la admisibilidad es menor del 90 %, no se publican intervalos posteriores Monte Carlo.** Un colapso dice que este conjunto y este prior no resuelven el posterior; no demuestra imposibilidad de estimar. Una alta proporción de estados negativos expone un límite físico del modelo bajo la incertidumbre evaluada.

También simulamos la propuesta de media logarítmica de la actualización local y comparamos la salida no lineal con su predicción lineal. Una discrepancia RMS mayor que una sigma de CO₂ alerta de extrapolación. Se exportan además residuos en muestras Y15 y Anton Paar que no se han asimilado, usando sólo CO₂ disponible hasta cada muestra. La biomasa no se usa para decidir cobertura. Este contraste evalúa una aproximación local, **no** un EKF validado ni un reajuste global.

La reponderación Monte Carlo usa todos los datos hasta el final del lote: es un diagnóstico **retrospectivo al final del horizonte**, y los cuantiles posteriores exportados, si se admite ESS, condicionan toda la trayectoria a ese conjunto. No se confunden con las covarianzas locales calculadas usando prefijos temporales.

**Cómo interpretar fallos por azúcar negativa:** en `base.rhs`, el consumo asociado al crecimiento contiene −qXG·growth_factor·X y −qXF·growth_factor·X. Esos términos pueden seguir siendo negativos con G=0 o F=0 si hay biomasa y nitrógeno disponible. Por ello la no negatividad de azúcares no está garantizada en todo el dominio de incertidumbre. Los controles registran el estado y su mínimo, rechazan valores menores que −10⁻⁷ g/L y sólo corrigen redondeos dentro de esa tolerancia. Se conserva la ecuación original; el diagnóstico expone su límite y no lo oculta mediante recortes.

In [ ]:
sobol=qmc.Sobol(d=len(ETA_NAMES),scramble=True,seed=U_CONFIG['seed'])
U_DRAWS=norm.ppf(np.clip(sobol.random_base2(U_CONFIG['mc_power']),1e-10,1-1e-10))
U_MC={};U_MC_STATUS=[];U_LINEARITY=[];U_MC_CHECK=[]
for (theta_name,lab),case in U_CASES.items():
    started=time.perf_counter();sd=prior_sd(lab,'referencia');draws=[];accepted=[];failures=[]
    cached=SIMULATION_CACHE.get((theta_name,lab))
    if cached is not None and np.array_equal(cached['draws'],U_DRAWS):
        draws=list(cached['ensemble']);accepted=list(cached['accepted'])
        failures=json.loads(cached['status'].iloc[0].failures)
        assert len(draws)+len(failures)==len(U_DRAWS)
        print(theta_name,lab,'Monte Carlo reutilizado: mismos draws y drivers',flush=True)
    else:
        for i,u in enumerate(U_DRAWS):
            try:
                frame=uncertainty_forward(lab,theta_name,u*sd,case['gate'])
                arr=frame[list(U_OUTPUTS)].to_numpy()
                if not np.isfinite(arr).all():raise ValueError('nonfinite')
                draws.append(arr);accepted.append(i)
            except (AssertionError,ValueError,RuntimeError) as error:
                failures.append(dict(draw=i,error=str(error)[:160]))
    assert draws,'No hay ninguna trayectoria admisible; revisar el dominio físico del modelo'
    ensemble=np.stack(draws);domain_adequate=len(ensemble)>=.90*len(U_DRAWS)
    prior_quantiles=np.quantile(ensemble,[.025,.50,.975],axis=0)
    ymc=np.stack([np.interp(case['times'],case['t'],arr[:,-1]) for arr in ensemble])
    U_MC[theta_name,lab]=dict(prior=prior_quantiles,ensemble=ensemble,accepted=accepted,posterior={})
    for mode in ('CO2','CO2_Y15'):
        logw=-.5*np.sum(((ymc-case['y'])/SIGMA[lab])**2,axis=1)
        if mode=='CO2_Y15':
            sentinel=SENTINELS[lab]
            for s in Y15_CHANNELS:
                values=[]
                for arr in ensemble:
                    if s=='GF': val=np.interp(sentinel.t_h,case['t'],arr[:,U_STATES.index('G')]+arr[:,U_STATES.index('F')])
                    else:val=np.interp(sentinel.t_h,case['t'],arr[:,U_STATES.index(s)])
                    values.append(val)
                observed=float(sentinel[Y15_COLS[s]]);sig=measurement_sigma(s,observed)
                logw-=.5*((np.asarray(values)-observed)/sig)**2
        weights=np.exp(logw-logsumexp(logw));ess=1./np.sum(weights**2)
        valid=bool(domain_adequate and ess>=max(U_CONFIG['ESS_min'],U_CONFIG['ESS_fraction_min']*len(weights)))
        U_MC_STATUS.append(dict(theta=theta_name,lab=lab,mode=mode,n_valid=len(ensemble),n_failed=len(failures),
                               prior_domain_adequate=domain_adequate,valid_fraction=len(ensemble)/len(U_DRAWS),
                               ESS=ess,max_weight=float(weights.max()),posterior_intervals_allowed=valid,failures=json.dumps(failures)))
        if valid:
            # Weighted quantiles, not interpolating weights against unsorted trajectories.
            qout=np.empty_like(prior_quantiles)
            for k in range(ensemble.shape[1]):
                for j in range(ensemble.shape[2]):
                    order=np.argsort(ensemble[:,k,j]);cum=np.cumsum(weights[order])
                    qout[:,k,j]=np.interp([.025,.5,.975],cum,ensemble[order,k,j])
            U_MC[theta_name,lab]['posterior'][mode]=qout
    for ti in (24.,48.,96.,case['t'][-1]):
        x,B=at_time(case,ti);linear_sd=np.linalg.norm(B*sd,axis=1)
        for k,s in enumerate(U_STATES):
            vals=np.array([np.interp(ti,case['t'],arr[:,k]) for arr in ensemble]);mc_sd=float(vals.std(ddof=1))
            U_MC_CHECK.append(dict(theta=theta_name,lab=lab,t_h=ti,state=s,linear_sigma=linear_sd[k],
                prior_domain_adequate=domain_adequate,valid_fraction=len(ensemble)/len(U_DRAWS),
                MC_sigma=mc_sd,ratio_MC_to_linear=mc_sd/linear_sd[k] if linear_sd[k]>1e-10 else np.nan,
                MC_q025=float(np.quantile(vals,.025)),MC_median=float(np.median(vals)),MC_q975=float(np.quantile(vals,.975))))
    for mode in ('CO2','CO2_Y15'):
        C,mean,sd=U_POST[theta_name,lab,'referencia',mode];eta=sd*mean
        max_eta=float(np.max(np.abs(eta)));linearity_error=np.nan;reason=''
        try:
            if max_eta>3.:raise ValueError('proposal log displacement > 3; no reliable local extrapolation')
            proposal=uncertainty_forward(lab,theta_name,eta,case['gate'])
            linear=case['pred']+case['J']@eta
            nonlinear=np.interp(case['times'],case['t'],proposal.predicted)
            linearity_error=float(np.sqrt(np.mean((nonlinear-linear)**2)))/SIGMA[lab]
        except (AssertionError,ValueError,RuntimeError) as error:reason=str(error)[:200]
        U_LINEARITY.append(dict(theta=theta_name,lab=lab,mode=mode,max_log_shift=max_eta,
            max_shift_prior_sd=float(np.max(np.abs(mean))),nonlinear_vs_linear_RMS_over_sigma=linearity_error,
            local_update_supported=bool(np.isfinite(linearity_error) and linearity_error<=1.),reason=reason))
    np.savez_compressed(OUT/f'uncertainty_MC_checkpoint_{theta_name}_{lab}.npz',ensemble=ensemble,accepted=np.asarray(accepted),prior=prior_quantiles)
    (OUT/'uncertainty_MC_checkpoint.json').write_text(json.dumps(dict(status=U_MC_STATUS,linearity=U_LINEARITY,MC_check=U_MC_CHECK,input_sha256=HASH_BEFORE)),encoding='utf-8')
    print(theta_name,lab,'Monte Carlo:',len(ensemble),'simulaciones,',round(time.perf_counter()-started,1),'s',flush=True)
U_MC_STATUS_TABLE=pd.DataFrame(U_MC_STATUS);U_LINEARITY_TABLE=pd.DataFrame(U_LINEARITY)
display(U_MC_STATUS_TABLE.drop(columns='failures').round(3))
display(U_LINEARITY_TABLE.round(3))
display(U_VALID_TABLE.groupby(['theta','lab','mode']).agg(n=('inside_local95','size'),
    fraction_inside_local95=('inside_local95','mean'),RMS_standardized=('standardized_residual',lambda s:np.sqrt(np.mean(s**2)))).round(3))
display(Markdown('**Lectura obligatoria:** las fracciones anteriores contrastan intervalos de la aproximación local. '
    'Si la comprobación no lineal falla, no respaldan una incertidumbre posterior fiable. '
    'Los resultados Monte Carlo con ESS insuficiente quedan suprimidos, no se dibujan como bandas posteriores.'))

### 11.3 Comprobar también la linealización de la química

Pasar una comprobación de CO₂ no basta para aceptar la actualización con Y15: una modificación de γ puede conservar exactamente la señal gaseosa y cambiar mucho Gly. Por eso evaluamos la propuesta de media local con el modelo no lineal también en N, G, G+F y Gly de la muestra asimilada. Se compara el cambio entre predicción lineal y no lineal con la sigma supuesta de cada canal. La comprobación conjunta exige RMS gaseoso ≤1 sigma y discrepancia máxima química ≤1 sigma. Es un control de validez de la aproximación, no una validación contra muestras independientes ni una garantía del EKF.


In [ ]:
U_CHEM_LINEARITY=[];U_CHEM_CHANNELS=[]
for (theta_name,lab),case in U_CASES.items():
    for mode in ('CO2','CO2_Y15'):
        old=U_LINEARITY_TABLE[U_LINEARITY_TABLE.theta.eq(theta_name)&U_LINEARITY_TABLE.lab.eq(lab)&U_LINEARITY_TABLE['mode'].eq(mode)].iloc[0]
        C,mean,sd=U_POST[theta_name,lab,'referencia',mode];eta=sd*mean
        chemical_error=np.nan;reason='';nchem=0
        try:
            if np.max(np.abs(eta))>3.:raise ValueError('log displacement > 3; local extrapolation unsupported')
            proposal=uncertainty_forward(lab,theta_name,eta,case['gate'])
            if mode=='CO2_Y15':
                sentinel=SENTINELS[lab];ti=float(sentinel.t_h);x,B=at_time(case,ti)
                predicted_linear=x+B@eta;errors=[]
                for state in Y15_CHANNELS:
                    if state=='GF':
                        linear=predicted_linear[U_STATES.index('G')]+predicted_linear[U_STATES.index('F')]
                        nonlinear=np.interp(ti,case['t'],proposal.G+proposal.F)
                    else:
                        linear=predicted_linear[U_STATES.index(state)]
                        nonlinear=np.interp(ti,case['t'],proposal[state])
                    observed=float(sentinel[Y15_COLS[state]])
                    error=(nonlinear-linear)/measurement_sigma(state,observed)
                    errors.append(abs(error))
                    U_CHEM_CHANNELS.append(dict(theta=theta_name,lab=lab,mode=mode,state=state,t_h=ti,
                        observed=observed,linear_prediction=linear,nonlinear_prediction=nonlinear,
                        nonlinear_minus_linear_over_sigma=error))
                chemical_error=float(max(errors));nchem=len(errors)
        except (AssertionError,ValueError,RuntimeError) as error:reason=str(error)[:200]
        gas_ok=bool(old.local_update_supported)
        chem_ok=bool(np.isfinite(chemical_error) and chemical_error<=1.) if mode=='CO2_Y15' else None
        joint_ok=bool(not reason and gas_ok and (chem_ok if mode=='CO2_Y15' else True))
        U_CHEM_LINEARITY.append(dict(theta=theta_name,lab=lab,mode=mode,gas_linearization_supported=gas_ok,
            n_chemical_channels_checked=nchem,max_chemical_discrepancy_over_sigma=chemical_error,
            chemical_linearization_supported=chem_ok,joint_linearization_supported=joint_ok,reason=reason))
U_CHEM_LINEARITY_TABLE=pd.DataFrame(U_CHEM_LINEARITY)
U_CHEM_LINEARITY_TABLE.to_csv(OUT/'uncertainty_chemistry_linearity.csv',index=False)
pd.DataFrame(U_CHEM_CHANNELS).to_csv(OUT/'uncertainty_chemistry_linearity_channels.csv',index=False)
display(U_CHEM_LINEARITY_TABLE)
display(pd.DataFrame(U_CHEM_CHANNELS))
display(Markdown(f'**Control conjunto:** {int(U_CHEM_LINEARITY_TABLE.joint_linearization_supported.sum())}/'
    f'{len(U_CHEM_LINEARITY_TABLE)} propuestas cumplen los criterios de linealización de todos sus canales asimilados. '
    'Las propuestas rechazadas no permiten interpretar sus bandas locales como incertidumbre posterior calibrada.'))

### 11.4 Cómo leer las figuras de esta ampliación

1. **CO₂ y dos theta:** puntos negros son mediciones; líneas, predicciones congeladas; zonas transparentes, cuantiles 2.5–97.5 % de las **simulaciones no lineales admisibles** del prior de referencia. Si muchas simulaciones fallan, es un prior truncado por admisibilidad, no el prior completo. No son bandas posteriores. La línea vertical marca la muestra Y15 ancla; la punteada, el pulso de nutrición. Ambas variantes usan los parámetros antiguos.
2. **Incertidumbre restante de estados:** σ después / σ antes. 1 significa que el CO₂ o la muestra apenas reducen la incertidumbre; 0.5 significa mitad de desviación estándar. Línea continua: sólo CO₂; discontinua: CO₂+la muestra Y15 desde que existe. Los colores distinguen theta. Cada punto usa sólo datos disponibles hasta ese momento. Estas curvas expresan información local potencial, incluso cuando la media lineal falla.
3. **Bandas de estados locales:** se muestran los anchos ±1.96σ **centrados en el nominal**, para separar la reducción de incertidumbre potencial del desplazamiento de la estimación. Se comparan prior, CO₂ y CO₂+Y15. Los puntos son química reservada; diamante, la muestra asimilable. **No son intervalos posteriores calibrados.** Si el nominal es sesgado, una banda estrecha no arregla ese sesgo. La tabla de contraste sí incluye el desplazamiento de la media lineal.
4. **Robustez:** comparar priors alto/medio/bajo, cadencia 6 h, ruido doble y una ganancia gaseosa residual constante incierta. La última se añade como control conservador sobre la ganancia ya incierta; no es una medición adicional.

Los intervalos gaussianos locales no se recortan artificialmente en cero: si incluyen valores negativos, muestran una limitación de esa aproximación. Las simulaciones no lineales conservan estados físicos no negativos.

**Los huecos en la curva de N no significan incertidumbre cero demostrada:** cuando σ previa ≤10⁻¹², el cociente posterior/previa se guarda como NaN y no se dibuja. En esta trayectoria nominal el nitrógeno se agota y sus derivadas locales pueden hacerse prácticamente nulas. Esa degeneración de la aproximación local no permite concluir que el nitrógeno real se conozca exactamente; hay que contrastarla con las muestras reservadas y la propagación no lineal. Las bandas locales centradas en N≈0 tampoco demuestran agotamiento experimental.

Las estrellas violetas de N son NUT; la barra horizontal indica su intervalo temporal provisional. Se conservan separadas del diamante del ancla Y15.


In [ ]:
THETA_COLOR=dict(theta_full='#2166ac',theta_defaults='#d6604d')
fig,axes=plt.subplots(3,1,figsize=(13,10),layout='constrained')
for ax,lab in zip(axes,LABS):
    obs=co2_hourly[co2_hourly.lab.eq(lab)&co2_hourly.co2_valid]
    ax.scatter(obs.t_h,obs.co2_observed_g_l_h,c='black',s=8,alpha=.5,label='CO₂ observado')
    for name in THETA_SETS:
        case=U_CASES[name,lab];q=U_MC[name,lab]['prior'];color=THETA_COLOR[name]
        ax.fill_between(case['t'],q[0,:,-1],q[2,:,-1],color=color,alpha=.12)
        ax.plot(case['t'],case['nominal'].predicted,color=color,label=name+' · nominal; banda MC admisible')
    ax.axvline(SENTINELS[lab].t_h,color='purple',ls='--',label='Muestra Y15 ancla')
    ax.axvline(PULSE2_H[lab],color='grey',ls=':',label='Pulso N')
    ax.set(title=lab+' · parámetros antiguos · comparación predictiva',xlabel='t [h]',ylabel='CO₂ [g/L/h]')
axes[0].legend(fontsize=8,ncol=2);display_inline_figure(fig)
fig,axes=plt.subplots(3,5,figsize=(17,10),layout='constrained')
for i,lab in enumerate(LABS):
    for j,state in enumerate(('X','N','G','F','E')):
        ax=axes[i,j]
        for name in THETA_SETS:
            for mode,ls in (('CO2','-'),('CO2_Y15','--')):
                rows=U_INTERVAL_TABLE.query('theta==@name and lab==@lab and scenario=="referencia" and state==@state and mode==@mode').sort_values('t_h')
                ax.plot(rows.t_h,rows.sigma_ratio,ls,color=THETA_COLOR[name],label=name+' '+mode)
        ax.axhline(1.,color='grey',lw=.7);ax.axvline(SENTINELS[lab].t_h,color='purple',ls=':',lw=.8)
        ax.set(title=f'{lab} · {state}',xlabel='t [h]',ylabel='σ restante / σ previa',ylim=(-.03,1.05))
axes[0,0].legend(fontsize=6);display_inline_figure(fig)
for name in THETA_SETS:
    fig,axes=plt.subplots(3,4,figsize=(16,10),layout='constrained')
    for i,lab in enumerate(LABS):
        for j,state in enumerate(('N','G','F','E')):
            ax=axes[i,j]
            for mode,color,alpha in (('prior','grey',.13),('CO2','#2166ac',.16),('CO2_Y15','#1b9e77',.20)):
                rows=U_INTERVAL_TABLE.query('theta==@name and lab==@lab and scenario=="referencia" and state==@state and mode==@mode').sort_values('t_h')
                ax.fill_between(rows.t_h,rows.nominal-1.96*rows.sigma,rows.nominal+1.96*rows.sigma,color=color,alpha=alpha,label=mode+' · ancho local')
            ax.plot(rows.t_h,rows.nominal,color='black',lw=1,label='Nominal congelado')
            col=OBS_COLUMNS[state];support=chem[chem.lab.eq(lab)&~chem.is_initial&chem[col].notna()]
            if state in ('G','F'):support=support[~support.soft_flag_GF]
            held=support[~support.is_NUT&~support['sample'].astype(str).eq(str(SENTINELS[lab]['sample']))]
            ax.scatter(held.t_h,held[col],c='black',s=12,label='Mediciones reservadas')
            draw_nut_state(ax,lab,state)
            sentinel=SENTINELS[lab];ax.scatter([sentinel.t_h],[sentinel[col]],c='purple',marker='D',s=24,label='Muestra ancla')
            ax.set(title=f'{lab} · {state} · {name}',xlabel='t [h]',ylabel=UNITS[state])
    axes[0,0].legend(fontsize=6);display_inline_figure(fig)
display(U_INTERVAL_TABLE.query('t_h==96 and state in ["X","N","G","F","E"]')
        .groupby(['theta','scenario','mode']).sigma_ratio.median().unstack('mode').round(3))
display(pd.DataFrame(U_STRESS).groupby(['theta','cadence_h','noise_factor','gas_constant_gain_nuisance_cv']).sigma_ratio.median().round(3))

### 11.5 Conclusión y decisiones para el EKF

Las conclusiones numéricas de la celda siguiente se generan a partir de esta ejecución. Se deben leer en este orden: **adecuación del nominal → validez local/no lineal → reducción de incertidumbre → contraste reservado**. Una reducción matemática por sí sola no justifica estimar simultáneamente todos los estados y parámetros.

La prueba permite elegir qué cantidades requieren una medición inicial más precisa, qué parámetros mantener calibrados y qué aporta una muestra Y15. No certifica la exactitud del CO₂ actual. Las bandas excluyen error estructural, incertidumbre de temperatura, dosis y conversiones, y dependen de priors sin calibración metrológica. El reservorio disuelto, O₂, gate y memoria de nutrición necesitan validación específica antes de trasladar estas conclusiones a un EKF operativo.

In [ ]:
U_FIT_TABLE=pd.DataFrame(U_POINT_FIT)
for theta_name in THETA_SETS:
    fit=U_FIT_TABLE[U_FIT_TABLE.theta.eq(theta_name)]
    display(Markdown(f'**{theta_name}, capa antigua:** RMSE CO₂={fit.RMSE_CO2.min():.3f}–{fit.RMSE_CO2.max():.3f} g/L/h '
        f'({fit.RMSE_over_sigma.min():.1f}–{fit.RMSE_over_sigma.max():.1f} veces la sigma supuesta). '
        f'Cambiar Cd₀ de 0 a 0.30 cambia como máximo {fit.Cd0_effect_zero_vs_030.max():.3f} g/L/h en el soporte asimilado.'))
    ratios=U_INTERVAL_TABLE.query('theta==@theta_name and scenario=="referencia" and t_h==96')
    display(ratios.pivot_table(index=['lab','state'],columns='mode',values='sigma_ratio').round(3))
allowed=int(U_MC_STATUS_TABLE.posterior_intervals_allowed.sum())
supported=int(U_LINEARITY_TABLE.local_update_supported.sum())
display(Markdown(f'**Comprobación práctica:** {allowed}/{len(U_MC_STATUS_TABLE)} posteriores Monte Carlo superan el umbral ESS; '
    f'{supported}/{len(U_LINEARITY_TABLE)} propuestas locales pasan la comprobación no lineal de una sigma de CO₂. '
    'Si estas comprobaciones fallan, la reducción de σ es sólo información potencial alrededor del nominal, '
    'y no una afirmación de precisión alcanzable con los datos reales.'))
display(Markdown('**Qué llevar al EKF:** conservar priors de química inicial, no fijar X₀ como conocido, '
    'calibrar la capa gaseosa antes de liberar simultáneamente ganancias y cinética, y contrastar con muestras Y15 reservadas. '
    'La comparación entre theta debe valorar también química y validez no lineal; un RMSE gaseoso menor por sí solo no elige el mejor modelo. '
    '**Qué falta medir:** incertidumbres y covarianzas instrumentales, biomasa independiente confiable y, si es posible, CO₂ disuelto. '
    'No se concluye observabilidad global ni se ha validado un EKF.'))
exports={'uncertainty_priors':PRIOR_TABLE,'uncertainty_intervals':U_INTERVAL_TABLE,
    'uncertainty_weak_directions':pd.DataFrame(U_WEAK),'uncertainty_correlations':pd.DataFrame(U_CORR),
    'uncertainty_validation':U_VALID_TABLE,'uncertainty_stress':pd.DataFrame(U_STRESS),
    'uncertainty_MC_status':U_MC_STATUS_TABLE,'uncertainty_MC_vs_linear':pd.DataFrame(U_MC_CHECK),
    'uncertainty_linearity':U_LINEARITY_TABLE,'uncertainty_theta_fit':U_FIT_TABLE,
    'uncertainty_FD_checks':pd.DataFrame(U_FD),
    'uncertainty_FD_covariance_impact':pd.DataFrame(U_FD_IMPACT),
    'uncertainty_Y15_anchor':pd.DataFrame([{**s.to_dict(),'lab':lab} for lab,s in SENTINELS.items()])}
for name,table in exports.items():table.to_csv(OUT/f'{name}.csv',index=False)
arrays={}
for i,((theta_name,lab),case) in enumerate(U_CASES.items()):
    case['nominal'].rename_axis('t_h').to_csv(OUT/f'uncertainty_nominal_{theta_name}_{lab}.csv')
    arrays[f'D_{i}']=case['D'];arrays[f't_{i}']=case['t'];arrays[f'prior_quantiles_{i}']=U_MC[theta_name,lab]['prior']
    arrays[f'Dhalf_{i}']=case['Dhalf']
    arrays[f'mc_draws_{i}']=U_MC[theta_name,lab]['ensemble'];arrays[f'accepted_draws_{i}']=np.asarray(U_MC[theta_name,lab]['accepted'])
    for mode,quantiles in U_MC[theta_name,lab]['posterior'].items():arrays[f'posterior_quantiles_{i}_{mode}']=quantiles
arrays['standard_normal_draws']=U_DRAWS
np.savez_compressed(OUT/'uncertainty_arrays.npz',**arrays)
assert HASH_BEFORE=={str(p.relative_to(ROOT)):sha256(p) for p in PROTECTED}
u_manifest=dict(config=U_CONFIG,scenarios=PRIOR_SCENARIOS,chem_sigma=CHEM_SIGMA,unknowns=ETA_NAMES,
    outputs=U_OUTPUTS,theta_sets=THETA_SETS,co2_parameters=CO2_PARAMS,
    old_parameters_verified_against_notebook02=True,biomass_validation=False,
    case_index={str(i):dict(theta=k[0],lab=k[1]) for i,k in enumerate(U_CASES)},
    initial_Cd_is_hypothesis=True,prior_covariance='independent log coordinates; F=GF-G',
    input_sha256=HASH_BEFORE,posterior_MC_accepted=allowed,local_updates_supported=supported,
    optimizer_calls=0,EKF_implemented=False,model_error_in_covariance=False,
    NUT_samples_included=3,NUT_assimilated=False,
    NUT_time_is_provisional=bool(NUT_SAMPLES.time_provisional.any()),
    NUT_parser_sha256=sha256(LAB2026/'nut_sample_audit.py'),
    forward_code_sha256=FORWARD_CODE_SHA256,simulation_cache_guard='hashes/settings/function AST/grid/nominal/anchor')
(OUT/'uncertainty_manifest.json').write_text(json.dumps(u_manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print('VERIFICADO: dos theta × tres lotes; parámetros antiguos; priors; muestras reservadas; sin biomasa en validación; entradas intactas.')
print('Resultados ampliados:',OUT)

# Contraste NUT con el núcleo continuo y los saltos exactos, no interpolando sobre el salto.
NUT_VALIDATION=[]
fig,axes=plt.subplots(3,1,figsize=(12,9),layout='constrained')
for ax,lab in zip(axes,LABS):
    nut=NUT_SAMPLES[NUT_SAMPLES.lab.eq(lab)].iloc[0]
    lo=float(nut.time_lower_h);hi=float(nut.time_upper_h)
    # El límite inferior es post-pulso: la muestra coincidente oficial es pre-acción.
    right_lo=max(lo,PULSE2_H[lab]+1e-6);right_hi=max(hi,right_lo)
    evaluation_time=max(float(nut.plot_t_h),right_lo)
    times=np.unique(np.r_[0.,np.linspace(right_lo,right_hi,25),evaluation_time])
    times=times[times>=0.]
    for name,th in THETA_SETS.items():
        core=integrate_core(recent_batches[lab],th,times)
        interval=core[core.index>PULSE2_H[lab]]
        prediction=float(np.interp(evaluation_time,core.index,core.N))
        NUT_VALIDATION.append(dict(theta=name,lab=lab,sample='NUT',
            time_provisional=bool(nut.time_provisional),time_lower_h=lo,time_upper_h=hi,
            plotted_t_h=float(nut.plot_t_h),observed_mg_L=float(nut.YAN_mg_l),
            reported_YAN_mg_L=float(nut.YAN_reported_mg_l),
            nominal_midpoint_mg_L=1000*prediction,
            nominal_min_over_interval_mg_L=1000*float(interval.N.min()),
            nominal_max_over_interval_mg_L=1000*float(interval.N.max()),
            residual_observed_minus_nominal_min_mg_L=float(nut.YAN_mg_l)-1000*float(interval.N.max()),
            residual_observed_minus_nominal_max_mg_L=float(nut.YAN_mg_l)-1000*float(interval.N.min()),
            used_as_assimilation_anchor=False, used_in_exact_time_coverage=False))
        curve=U_CASES[name,lab]['nominal']
        ax.plot(curve.index,1000*curve.N,color=THETA_COLOR[name],label=name)
    observed=chem[chem.lab.eq(lab)&~chem.is_NUT]
    ax.scatter(observed.t_h,1000*observed.N_kg_m3,color='black',s=20,label='YAN muestras numeradas')
    x=float(nut.plot_t_h)
    ax.errorbar([x],[nut.YAN_mg_l],xerr=[[x-lo],[hi-x]],fmt='*',color='#8e24aa',
                ms=10,capsize=3,label='NUT · tiempo provisional' if nut.time_provisional else 'NUT · hora confirmada')
    ax.axvline(PULSE2_H[lab],color='grey',ls=':',label='Pulso 2 registrado')
    ax.set(title=lab+' · NUT como contraste químico reservado',xlabel='t desde inoculación [h]',ylabel='YAN [mg N/L]')
axes[0].legend(fontsize=8,ncol=2)
display_inline_figure(fig)
NUT_VALIDATION_TABLE=pd.DataFrame(NUT_VALIDATION)
NUT_VALIDATION_TABLE.to_csv(OUT/'nut_reserved_nitrogen_validation.csv',index=False)
display(NUT_VALIDATION_TABLE.round(4))
display(Markdown('**NUT incorporada:** tres muestras con PAN/amonio/YAN, sin repetir la dosis ni '
    'crear otros canales. Los límites temporales son provisionales, no intervalos estadísticos. '
    'La tabla evalúa el N nominal después del salto exacto sobre todo ese intervalo; '
    'NUT no se convierte en ancla ni se incluye en una cobertura que requiera hora exacta.'))


## Referencias y decisiones de interpretación

- Fleurial, Sacchelli y Yabo (ECC 2024), sección III-A y apéndice: prueba para otro modelo y otra
  función de observación. Aquí no se reutiliza su conclusión de observabilidad.
- Lima et al. (preprint 2024), secciones 2.4–2.5: estimación y propagación de incertidumbre con azúcar
  como medición. Este notebook prepara la evaluación de información, no reproduce ese EKF.
- Auditoría térmica y comparación de nutrición locales: origen de ecuaciones simplificadas,
  parámetros, lectores y convenciones; hashes en el manifiesto.

**Siguiente decisión que permite este diagnóstico:** seleccionar estados/priors y mediciones útiles,
y definir una transición causal cerrada antes de construir y validar un EKF. Las muestras offline
reservadas servirán entonces para comprobar los estados estimados, no sólo el ajuste del CO₂.